# Explainable GeoAI assessment of wheat heat-stress livelihood risk in Pakistan

Complete analysis notebook for the paper *"Explainable GeoAI Locates Structural Wheat Heat-Stress Risk in
Pakistan: Integrating Earth Observation, Climate Projections and Socioeconomic Data"*.

The notebook follows the seven stages of the methodology:

| Section | Stage | Content |
|---|---|---|
| 1 | - | Setup, district yield target (Theil-Sen detrended anomalies) |
| 2 | A | Wheat-area-weighted extraction of ERA5-Land, CHIRPS and MODIS to districts; stage-wise predictors |
| 3 | A | XGBoost, random forest, ridge and trend baseline under three blocked cross-validation designs; Diebold-Mariano tests |
| 4 | A | SHAP attribution, stage-wise heat thresholds, accumulated local effects |
| 5 | A | Heat-attributable losses and hazard, present climate and 1.5/2/3 degC warming (35 NEX-GDDP-CMIP6 models) |
| 6 | B, C | Exposure; PCA-weighted vulnerability index and its validation |
| 7 | D | Livelihood risk with Monte Carlo uncertainty |
| 8 | E | Local Moran's I hotspots; concentration index of losses across the wealth distribution |
| 9 | F | SHAP on a surrogate of the risk index; district policy typology |
| 10 | G | Out-of-sample test against the 2021-22 (March-April 2022) heatwave season |
| 11 | - | Figures and tables for the manuscript |

All inputs are open data stored under `data_raw/` (see `data_raw/README.md`). Expensive intermediate
products (daily district climate series, CMIP6 deltas) are cached in `data_processed/analysis/`; delete
that folder to recompute everything from the raw files.

In [ ]:
import glob
import json
import os
import re
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
from rasterio.windows import from_bounds
from scipy import sparse, stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path(os.environ.get("PAPER8_ROOT", ".."))  # the notebook runs in analysis/
RAW = ROOT / "data_raw"
CACHE = ROOT / "data_processed" / "analysis"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "figures"
for p in (CACHE, TAB, FIG):
    p.mkdir(parents=True, exist_ok=True)

SEED = 20260514
rng_global = np.random.default_rng(SEED)
N_JOBS = max(1, os.cpu_count() - 2)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(N_JOBS))  # avoids a harmless loky core-count warning on Windows
BBOX = (60.0, 23.0, 79.0, 38.0)  # W, S, E, N
STUDY_PROVINCES = ["Punjab", "Sindh", "Khyber Pakhtunkhwa", "Balochistan", "Islamabad"]
HOLDOUT_SEASON = 2022  # 2021-22 season (harvest year 2022) is withheld from all model fitting

mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "font.size": 8,
    "axes.linewidth": 0.6, "axes.titlesize": 8, "axes.labelsize": 8, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "legend.fontsize": 7, "savefig.dpi": 300, "savefig.bbox": "tight",
    "xtick.major.width": 0.5, "ytick.major.width": 0.5, "pdf.fonttype": 42,
})
print("cores used:", N_JOBS)

## 1. Districts and the yield target

The district wheat panel (1981-82 to 2024-25, 2022 boundaries) was compiled from PBS, provincial Crop
Reporting Services, Bureaus of Statistics and MNFSR publications (`scripts/yield_panel/`). Only
district-own observations (`flag` = own / own_merged_units) enter the models; values inherited from a
parent district are kept for a robustness check. Seasons are indexed by harvest year (2021-22 -> 2022).

The technology trend of each district is a Theil-Sen line, re-centred so that the district's mean relative
anomaly over the fitting seasons is zero (equivalent to a district fixed effect).

In [ ]:
adm = gpd.read_file(RAW / "01_boundaries" / "pak_admin2.shp").to_crs(4326)
adm = adm[["adm2_pcode", "adm2_name", "adm1_name", "geometry"]]
study = adm[adm.adm1_name.isin(STUDY_PROVINCES)].reset_index(drop=True)
PCODES = study.adm2_pcode.tolist()
print("districts in study provinces:", len(study))

panel = pd.read_csv(ROOT / "data_processed" / "yield_panel" / "wheat_yield_district_panel_1982_2025.csv")
panel["own"] = panel.flag.str.startswith("own")
print(panel.flag.value_counts().to_string())

In [ ]:
def theil_sen_trend(years, values, fit_mask):
    """District technology trend: Theil-Sen slope/intercept fitted on fit_mask, evaluated on all years."""
    res = stats.theilslopes(values[fit_mask], years[fit_mask])
    return res.slope, res.intercept


MIN_YEARS = 12  # minimum district-own seasons (outside the hold-out) to fit a trend
rows = []
for pc, g in panel[panel.yield_t_ha.notna()].groupby("adm2_pcode"):
    g = g.sort_values("year")
    own = g.own.values
    fit = own & (g.year.values != HOLDOUT_SEASON)
    if fit.sum() < MIN_YEARS:
        continue
    slope, icpt = theil_sen_trend(g.year.values.astype(float), g.yield_t_ha.values, fit)
    trend = icpt + slope * g.year.values
    # centre the trend so that the mean relative anomaly over the fitting seasons is zero (district fixed
    # effect): otherwise skewed yield distributions leave district-specific offsets that location-proxy
    # predictors (absolute temperatures) can learn
    off = np.mean((g.yield_t_ha.values[fit] - trend[fit]) / trend[fit])
    trend = trend * (1 + off)
    slope = slope * (1 + off)
    rows.append(pd.DataFrame({"adm2_pcode": pc, "year": g.year.values, "yield_t_ha": g.yield_t_ha.values,
                              "trend_t_ha": trend, "own": own, "flag": g.flag.values,
                              "area_ha": g.area_ha.values, "slope_t_ha_yr": slope}))
tgt = pd.concat(rows, ignore_index=True)
tgt["anom"] = (tgt.yield_t_ha - tgt.trend_t_ha) / tgt.trend_t_ha
tgt = tgt[tgt.trend_t_ha > 0.1]

# districts that grow little wheat are noisy; keep those with a median own-season area >= 1,000 ha
med_area = tgt[tgt.own].groupby("adm2_pcode").area_ha.median()
keep = med_area[med_area >= 1000].index
tgt = tgt[tgt.adm2_pcode.isin(keep)].copy()
# implausible anomalies (data errors) are removed: |anomaly| > 75 % of trend
n_out = int((tgt.anom.abs() > 0.75).sum())
tgt = tgt[tgt.anom.abs() <= 0.75].copy()
tgt = tgt.merge(study[["adm2_pcode", "adm2_name", "adm1_name"]], on="adm2_pcode")
print(f"districts with a trend: {tgt.adm2_pcode.nunique()} | own obs: {tgt.own.sum()} | "
      f"inherited obs: {(~tgt.own).sum()} | removed |anom|>0.75: {n_out}")
print(tgt[tgt.own].anom.describe().round(3).to_string())
tgt.to_csv(CACHE / "yield_target.csv", index=False)

## 2. Climate and Earth-observation predictors (Stage A)

### 2.1 Wheat-area weights
Each district's climate is the average over its wheat area: every 5-arcmin SPAM2010 pixel inside a district
carries its wheat physical area as weight (plus 0.001 ha so that districts without mapped wheat fall back to
a uniform average), and is assigned to the climate grid cell that contains it.

In [ ]:
def read_spam(tech):
    with rasterio.open(RAW / "04_crop" / "spam2010" / f"spam2010V2r0_global_A_WHEA_{tech}.tif") as src:
        win = from_bounds(*BBOX, transform=src.transform)
        a = src.read(1, window=win).astype("float64")
        tr = src.window_transform(win)
    a[a < 0] = 0.0
    return a, tr


spam_a, spam_tr = read_spam("A")
spam_i, _ = read_spam("I")
r_idx, c_idx = np.indices(spam_a.shape)
px_lon = spam_tr.c + (c_idx + 0.5) * spam_tr.a
px_lat = spam_tr.f + (r_idx + 0.5) * spam_tr.e
pts = gpd.GeoDataFrame({"wheat": spam_a.ravel(), "wheat_irr": spam_i.ravel()},
                       geometry=gpd.points_from_xy(px_lon.ravel(), px_lat.ravel()), crs=4326)
pts = gpd.sjoin(pts, study[["adm2_pcode", "geometry"]], predicate="within", how="inner")
pts["lon"], pts["lat"] = pts.geometry.x, pts.geometry.y
pts["w"] = pts.wheat + 1e-3
didx = {pc: i for i, pc in enumerate(PCODES)}
pts["d"] = pts.adm2_pcode.map(didx)
print("SPAM pixels inside study districts:", len(pts), "| wheat area (Mha):", round(pts.wheat.sum() / 1e6, 2))

irr = pts.groupby("adm2_pcode")[["wheat", "wheat_irr"]].sum()
irr_share = (irr.wheat_irr / irr.wheat.replace(0, np.nan)).clip(0, 1).fillna(0).rename("irr_share")
spam_wheat_ha = irr.wheat.rename("spam_wheat_ha")


def weight_matrix(lat, lon):
    """Sparse (n_districts x n_cells) matrix mapping a (lat, lon) grid to wheat-weighted district means."""
    lat, lon = np.asarray(lat), np.asarray(lon)
    il = np.abs(pts.lat.values[:, None] - lat[None, :]).argmin(1)
    jl = np.abs(pts.lon.values[:, None] - lon[None, :]).argmin(1)
    cell = il * len(lon) + jl
    W = sparse.csr_matrix((pts.w.values, (pts.d.values, cell)), shape=(len(PCODES), len(lat) * len(lon)))
    return W


def district_mean(W, arr):
    """arr: (time, lat, lon) -> (time, n_districts), NaN-aware weighted mean."""
    X = arr.reshape(arr.shape[0], -1).T  # cells x time
    ok = np.isfinite(X)
    num = W @ np.where(ok, X, 0.0)
    den = W @ ok.astype("float64")
    return (num / np.where(den > 0, den, np.nan)).T

### 2.2 Daily district series (cached)
ERA5-Land daily Tmax/Tmin (0.1 deg), dew point and volumetric soil moisture (0-7 and 7-28 cm), and CHIRPS
daily rainfall (0.05 deg), 1980-11 to 2025-05.

In [ ]:
Y0, Y1 = 1980, 2025
CLIM = RAW / "02_climate"


def extract_var(name, files, var, conv=lambda a: a):
    out = CACHE / f"daily_{name}.parquet"
    if out.exists():
        return pd.read_parquet(out)
    W, parts = None, []
    for f in files:
        with xr.open_dataset(f, engine="h5netcdf") as ds:
            latn = "lat" if "lat" in ds.coords else "latitude"
            lonn = "lon" if "lon" in ds.coords else "longitude"
            if W is None:
                W = weight_matrix(ds[latn].values, ds[lonn].values)
            a = conv(ds[var].transpose("time", latn, lonn).values.astype("float64"))
            parts.append(pd.DataFrame(district_mean(W, a), index=pd.to_datetime(ds.time.values).normalize(),
                                      columns=PCODES))
    df = pd.concat(parts).sort_index()
    df = df[~df.index.duplicated()]
    df.to_parquet(out)
    return df


t0 = time.time()
K2C = lambda a: a - 273.15 if np.nanmean(a) > 200 else a
tmax = extract_var("tmax", [CLIM / "era5land_tmax" / f"era5land_tmax_{y}.nc" for y in range(Y0, Y1 + 1)], "tmax", K2C)
tmin = extract_var("tmin", [CLIM / "era5land_tmin" / f"era5land_tmin_{y}.nc" for y in range(Y0, Y1 + 1)], "tmin", K2C)
dfiles = sorted(glob.glob(str(CLIM / "era5land_dewpoint_soilmoisture" / "era5land_dew_swvl_PAK_*.nc")))
dew = extract_var("dew", dfiles, "dewpoint_temperature_2m", K2C)
sm1 = extract_var("swvl1", dfiles, "volumetric_soil_water_layer_1")
sm2 = extract_var("swvl2", dfiles, "volumetric_soil_water_layer_2")
prcp = extract_var("prcp", [CLIM / "chirps_v2_daily" / f"chirps-v2.0.{y}.days_p05_PAK.nc" for y in range(1981, Y1 + 1)],
                   "precip", lambda a: np.where(a < 0, np.nan, a))
sm = (sm1 * 7 + sm2 * 21) / 28  # 0-28 cm depth-weighted
print(f"daily series ready ({time.time() - t0:.0f} s):", tmax.shape, prcp.shape, "Tmax range",
      round(np.nanmin(tmax.values), 1), round(np.nanmax(tmax.values), 1))

### 2.3 Stage-wise predictors
Five stages from the provincial sowing calendars (sowing 1-20/25 Nov in Punjab and KP, Nov-Dec in Sindh):
sowing (Nov), tillering (Dec-Jan), jointing (Feb), anthesis (Mar) and grain filling (Apr). For each stage:
growing degree days (base 0 degC, capped at 30 degC), frost days (Tmin < 0 degC), mean vapour-pressure deficit,
rainfall, longest dry spell (< 1 mm/day), mean soil moisture, mean Tmax and killing degree days above
30 and 34 degC (Eq. 2).

In [ ]:
STAGES = {"sow": [(11, 1, 11, 30)], "till": [(12, 1, 12, 31), (1, 1, 1, 31)], "joint": [(2, 1, 2, 29)],
          "anth": [(3, 1, 3, 31)], "fill": [(4, 1, 4, 30)]}
STAGE_ORDER = list(STAGES)


def stage_labels(idx):
    m, d, y = idx.month.values, idx.day.values, idx.year.values
    season = np.where(m >= 11, y + 1, y)
    lab = np.full(len(idx), "", dtype=object)
    for s, wins in STAGES.items():
        for (m0, d0, m1, d1) in wins:
            sel = (m == m0) & (d >= d0) & (d <= d1) if m0 == m1 else np.zeros(len(idx), bool)
            lab[sel] = s
    return season, lab


def es(t):  # saturation vapour pressure (kPa), Tetens
    return 0.6108 * np.exp(17.27 * t / (t + 237.3))


def longest_run(mask2d):
    """Longest run of True along axis 0 for each column."""
    best = np.zeros(mask2d.shape[1], int)
    cur = np.zeros(mask2d.shape[1], int)
    for row in mask2d:
        cur = np.where(row, cur + 1, 0)
        best = np.maximum(best, cur)
    return best


def stage_features(tx, tn, td, pr, smx, seasons):
    """Return long table (adm2_pcode, year) x stage features from daily district frames."""
    idx = tx.index
    season, lab = stage_labels(idx)
    tmean = (tx + tn) / 2
    vpd = (es(tx) + es(tn)) / 2 - es(td)
    gdd = tmean.clip(lower=0, upper=30)
    out = []
    for yr in seasons:
        rec = {"year": np.full(len(PCODES), yr), "adm2_pcode": PCODES}
        for s in STAGE_ORDER:
            sel = (season == yr) & (lab == s)
            if sel.sum() == 0:
                continue
            X, N = tx.values[sel], tn.values[sel]
            rec[f"gdd_{s}"] = gdd.values[sel].sum(0)
            rec[f"frost_{s}"] = (N < 0).sum(0)
            rec[f"vpd_{s}"] = np.nanmean(vpd.values[sel], 0)
            rec[f"tmax_{s}"] = np.nanmean(X, 0)
            rec[f"kdd30_{s}"] = np.clip(X - 30, 0, None).sum(0)
            rec[f"kdd34_{s}"] = np.clip(X - 34, 0, None).sum(0)
            if pr is not None:
                P = pr.reindex(idx).values[sel]
                rec[f"rain_{s}"] = np.nansum(P, 0)
                rec[f"dry_{s}"] = longest_run(np.nan_to_num(P, nan=0) < 1.0)
            if smx is not None:
                rec[f"sm_{s}"] = np.nanmean(smx.reindex(idx).values[sel], 0)
        out.append(pd.DataFrame(rec))
    return pd.concat(out, ignore_index=True)


SEASONS = list(range(1982, 2026))
feat_path = CACHE / "features_obs.parquet"
if feat_path.exists():
    feat = pd.read_parquet(feat_path)
else:
    win = (tmax.index >= "1981-11-01") & (tmax.index <= "2025-04-30")
    feat = stage_features(tmax[win], tmin[win], dew.reindex(tmax.index[win]), prcp, sm, SEASONS)
    feat.to_parquet(feat_path)
feat = feat.merge(irr_share, left_on="adm2_pcode", right_index=True, how="left")
HEAT_VARS = [f"{v}_{s}" for s in STAGE_ORDER for v in ("tmax", "kdd30", "kdd34")]
feat["irr_x_kdd30_late"] = feat.irr_share * (feat.kdd30_anth + feat.kdd30_fill)
CLIM_FEATURES = [c for c in feat.columns if c not in ("year", "adm2_pcode")]
print("predictors:", len(CLIM_FEATURES))
print(feat[["tmax_anth", "tmax_fill", "kdd30_fill", "kdd34_fill", "rain_till", "sm_fill"]].describe().round(2).to_string())

### 2.4 MODIS NDVI/EVI (2001-2025, supplementary model)
District cropland means of 16-day MOD13Q1 composites (good-quality pixels, WorldCover cropland mask) are
summarised per season: mean NDVI in jointing (Feb), peak NDVI and mean EVI in anthesis-grain filling (Mar-Apr).
Because EO indices cannot be projected into the future, they enter only a supplementary model that measures
the added skill of Earth observation over climate alone.

In [ ]:
mod = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RAW / "03_earth_observation" /
                                                              "modis_mod13q1_ndvi_evi" / "*.csv")))])
mod["date"] = pd.to_datetime(mod.composite)
mod = mod[(mod.n_good_cropland_px >= 20) & mod.adm2_pcode.isin(PCODES)]
mod["m"] = mod.date.dt.month
mod["year"] = mod.date.dt.year
eo = pd.concat([
    mod[mod.m == 2].groupby(["adm2_pcode", "year"]).NDVI_mean.mean().rename("ndvi_joint"),
    mod[mod.m.isin([3, 4])].groupby(["adm2_pcode", "year"]).NDVI_mean.max().rename("ndvi_peak_late"),
    mod[mod.m.isin([3, 4])].groupby(["adm2_pcode", "year"]).EVI_mean.mean().rename("evi_late"),
], axis=1).reset_index()
EO_FEATURES = ["ndvi_joint", "ndvi_peak_late", "evi_late"]
print(eo.describe().round(3).to_string())

### 2.5 Modelling table

In [ ]:
data = tgt.merge(feat, on=["adm2_pcode", "year"], how="inner").merge(eo, on=["adm2_pcode", "year"], how="left")
data = data.dropna(subset=CLIM_FEATURES)
train_all = data[data.own & (data.year != HOLDOUT_SEASON)].reset_index(drop=True)
print(f"modelling rows (own, excl. 2022): {len(train_all)} | districts: {train_all.adm2_pcode.nunique()} | "
      f"seasons: {train_all.year.min()}-{train_all.year.max()} ({train_all.year.nunique()})")
data.to_parquet(CACHE / "model_table.parquet")

## 3. Hazard model and blocked cross-validation (Stage A)

Four predictors of the relative yield anomaly are compared: a trend-only baseline (anomaly = 0), ridge
regression, random forest and XGBoost. Hyperparameters are tuned in an inner loop (3-fold, grouped by
season). The district technology trends are refitted on the training seasons of every fold, so test
seasons never inform the detrending. Three outer designs avoid leakage:
leave-one-year-out (LOYO), leave-district-group-out (8 spatially contiguous k-means blocks), and forward
chaining (train on all seasons before a 5-year block, predict the block).

In [ ]:
import itertools

import xgboost as xgb
from joblib import Parallel, delayed
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if "data" not in globals():
    data = pd.read_parquet(CACHE / "model_table.parquet")
FEATS = CLIM_FEATURES
full = data[data.own].reset_index(drop=True)  # all own observations incl. 2022 (2022 used only in Stage G)
dev = full[full.year != HOLDOUT_SEASON].reset_index(drop=True)


def refit_anomaly(df, fit_mask):
    """Centred Theil-Sen trend per district on rows where fit_mask; returns anomaly and trend for all rows."""
    tr = np.full(len(df), np.nan)
    for pc, ix in df.groupby("adm2_pcode").indices.items():
        f = ix[fit_mask[ix]]
        if len(f) < 8:
            continue
        r = stats.theilslopes(df.yield_t_ha.values[f], df.year.values[f].astype(float))
        t_ = r.intercept + r.slope * df.year.values[ix]
        tf = r.intercept + r.slope * df.year.values[f]
        tr[ix] = t_ * (1 + np.mean((df.yield_t_ha.values[f] - tf) / tf))  # centred: zero mean anomaly on fit rows
    tr = np.where(tr > 0.1, tr, np.nan)
    return (df.yield_t_ha.values - tr) / tr, tr


GRIDS = {
    "Ridge": [{"alpha": a} for a in (1.0, 10.0, 100.0, 1000.0)],
    "RF": [{"max_features": mf, "min_samples_leaf": ml} for mf in (0.33, 0.6) for ml in (15, 40)],
    "XGB": [{"max_depth": md, "min_child_weight": mc} for md in (2, 3) for mc in (20, 60)],
}


def make_model(name, p, n_jobs=N_JOBS, seed=SEED):
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=p["alpha"]))
    if name == "RF":
        return RandomForestRegressor(n_estimators=300, max_features=p["max_features"],
                                     min_samples_leaf=p["min_samples_leaf"], n_jobs=n_jobs, random_state=seed)
    return xgb.XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=p["max_depth"],
                            min_child_weight=p["min_child_weight"], subsample=0.8, colsample_bytree=0.7,
                            reg_lambda=1.0, tree_method="hist", n_jobs=n_jobs, random_state=seed)


def tune(name, X, y, groups):
    gkf = GroupKFold(n_splits=3)
    best, best_mse = None, np.inf
    for p in GRIDS[name]:
        mse = []
        for a, b in gkf.split(X, y, groups):
            m = make_model(name, p).fit(X[a], y[a])
            mse.append(np.mean((m.predict(X[b]) - y[b]) ** 2))
        if np.mean(mse) < best_mse:
            best, best_mse = p, np.mean(mse)
    return best


def run_design(df, folds, label):
    """folds: list of (train_mask, test_mask, trend_mask). Returns out-of-fold predictions and params.
    trend_mask = seasons used to fit the district trends (training seasons for LOYO and forward chaining;
    all seasons of the held-out districts for the spatial design, whose test is spatial transfer)."""
    recs, chosen = [], []
    for k, (trm, tem, tdm) in enumerate(folds):
        anom, trend = refit_anomaly(df, tdm)
        ok = np.isfinite(anom)
        tr_i, te_i = np.where(trm & ok)[0], np.where(tem & ok)[0]
        if len(te_i) == 0:
            continue
        X, y = df[FEATS].values, anom
        pred = {"Trend": np.zeros(len(te_i))}
        for name in ("Ridge", "RF", "XGB"):
            p = tune(name, X[tr_i], y[tr_i], df.year.values[tr_i])
            chosen.append({"design": label, "fold": k, "model": name, **p})
            pred[name] = make_model(name, p).fit(X[tr_i], y[tr_i]).predict(X[te_i])
        recs.append(pd.DataFrame({"design": label, "fold": k, "adm2_pcode": df.adm2_pcode.values[te_i],
                                  "year": df.year.values[te_i], "obs": y[te_i], "trend": trend[te_i],
                                  "yield": df.yield_t_ha.values[te_i], **{f"pred_{m}": v for m, v in pred.items()}}))
    return pd.concat(recs, ignore_index=True), pd.DataFrame(chosen)


# district blocks for the spatial design
cent = study.set_index("adm2_pcode").to_crs(32642).geometry.centroid
dpc = dev.adm2_pcode.unique()
km = KMeans(n_clusters=8, n_init=20, random_state=SEED).fit(np.c_[cent[dpc].x, cent[dpc].y])
block = pd.Series(km.labels_, index=dpc)
dev["block"] = dev.adm2_pcode.map(block)

yrs = dev.year.values
FOLDS = {
    "LOYO": [(yrs != y, yrs == y, yrs != y) for y in sorted(dev.year.unique())],
    "Spatial blocks": [(dev.block.values != b, dev.block.values == b, np.ones(len(dev), bool)) for b in range(8)],
    "Forward chaining": [(yrs < a, (yrs >= a) & (yrs <= a + 4), yrs < a) for a in (2001, 2006, 2011, 2016, 2021)],
}
cv_path = CACHE / "cv_predictions.parquet"
if cv_path.exists():
    oof = pd.read_parquet(cv_path)
    params = pd.read_csv(CACHE / "cv_params.csv")
else:
    t0 = time.time()
    res = [run_design(dev, f, lab) for lab, f in FOLDS.items()]
    oof = pd.concat([r[0] for r in res], ignore_index=True)
    params = pd.concat([r[1] for r in res], ignore_index=True)
    oof.to_parquet(cv_path)
    params.to_csv(CACHE / "cv_params.csv", index=False)
    print(f"cross-validation done in {(time.time() - t0) / 60:.1f} min")

### 3.1 Skill and Diebold-Mariano tests
RMSE and R^2 of the anomaly, RMSE and relative RMSE of yield (anomaly converted back with the fold trend),
and the skill score over the trend baseline, SS = 1 - MSE_model / MSE_trend. Paired model comparisons use
the Diebold-Mariano test on the season series of district-mean squared-error differences (Newey-West
variance, lag 1, Harvey-Leybourne-Newbold small-sample correction).

In [ ]:
MODELS = ["Trend", "Ridge", "RF", "XGB"]


def dm_test(e1, e2, h=1):
    d = e1 - e2
    T = len(d)
    dc = d - d.mean()
    gamma = [np.sum(dc[k:] * dc[:T - k]) / T for k in range(h + 1)]
    var = gamma[0] + 2 * sum((1 - k / (h + 1)) * gamma[k] for k in range(1, h + 1))
    dm = d.mean() / np.sqrt(var / T)
    dm *= np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return dm, 2 * stats.t.sf(abs(dm), df=T - 1)


skill_rows, dm_rows = [], []
for des, g in oof.groupby("design", sort=False):
    mse_trend = np.mean(g.obs ** 2)
    for m in MODELS:
        e = g[f"pred_{m}"] - g.obs
        yhat = g.trend * (1 + g[f"pred_{m}"])
        rmse_y = np.sqrt(np.mean((yhat - g["yield"]) ** 2))
        skill_rows.append({"design": des, "model": m, "n": len(g), "RMSE_anom": np.sqrt(np.mean(e ** 2)),
                           "R2_anom": r2_score(g.obs, g[f"pred_{m}"]) if m != "Trend" else 0.0,
                           "RMSE_yield_t_ha": rmse_y, "rRMSE_yield_pct": 100 * rmse_y / g["yield"].mean(),
                           "R2_yield": r2_score(g["yield"], yhat), "SS_vs_trend": 1 - np.mean(e ** 2) / mse_trend})
    se = g.assign(**{f"se_{m}": (g[f"pred_{m}"] - g.obs) ** 2 for m in MODELS}).groupby("year")[
        [f"se_{m}" for m in MODELS]].mean()
    for a, b in [("XGB", "Trend"), ("RF", "Trend"), ("Ridge", "Trend"), ("XGB", "RF"), ("XGB", "Ridge"), ("RF", "Ridge")]:
        dm, p = dm_test(se[f"se_{a}"].values, se[f"se_{b}"].values)
        dm_rows.append({"design": des, "comparison": f"{a} vs {b}", "DM": dm, "p_value": p, "n_seasons": len(se)})
skill = pd.DataFrame(skill_rows)

# skill at aggregate scale (province-season, area-weighted) and by irrigation class (LOYO)
lo = oof[oof.design == "LOYO"].merge(dev[["adm2_pcode", "year", "area_ha", "adm1_name", "irr_share"]],
                                      on=["adm2_pcode", "year"])
lo["w"] = lo.area_ha.fillna(lo.area_ha.median())
agg_rows = []
for lev, keys in [("district-season", None), ("province-season", ["adm1_name", "year"]), ("national-season", ["year"])]:
    if keys is None:
        g_ = lo
    else:
        g_ = lo.groupby(keys).apply(lambda g: pd.Series({c: np.average(g[c], weights=g.w) for c in
                                                          ["obs"] + [f"pred_{m}" for m in MODELS]})).reset_index()
    for m in ("Ridge", "RF", "XGB"):
        agg_rows.append({"level": lev, "model": m, "n": len(g_), "r": stats.pearsonr(g_.obs, g_[f"pred_{m}"])[0],
                         "SS_vs_trend": 1 - np.mean((g_[f"pred_{m}"] - g_.obs) ** 2) / np.mean(g_.obs ** 2)})
lo["irr_class"] = np.where(lo.irr_share >= 0.8, "irrigated (>=80%)", np.where(lo.irr_share < 0.5, "rainfed (<50%)", "mixed"))
for cl, g_ in lo.groupby("irr_class"):
    for m in ("Ridge", "RF", "XGB"):
        agg_rows.append({"level": f"district-season, {cl}", "model": m, "n": len(g_),
                         "r": stats.pearsonr(g_.obs, g_[f"pred_{m}"])[0],
                         "SS_vs_trend": 1 - np.mean((g_[f"pred_{m}"] - g_.obs) ** 2) / np.mean(g_.obs ** 2)})
agg_skill = pd.DataFrame(agg_rows)
agg_skill.round(4).to_csv(TAB / "T2d_skill_by_scale_and_irrigation.csv", index=False)
print(agg_skill.round(3).to_string(index=False))
dmt = pd.DataFrame(dm_rows)
skill.round(4).to_csv(TAB / "T2_model_skill.csv", index=False)
dmt.round(4).to_csv(TAB / "T2b_diebold_mariano.csv", index=False)
print(skill.round(3).to_string(index=False))
print(dmt.round(3).to_string(index=False))

### 3.2 Final models and bootstrap ensemble
The final XGBoost and random forest are tuned on all development seasons (5-fold grouped by season) and
refitted 40 times on season-block bootstrap samples; the refits carry model uncertainty into the hazard and
the Monte Carlo risk analysis. The supplementary EO model adds MODIS NDVI/EVI (2001-2025).

In [ ]:
anom_dev, trend_dev = refit_anomaly(dev, np.ones(len(dev), bool))
dev["anom_fit"], dev["trend_fit"] = anom_dev, trend_dev
dv = dev[np.isfinite(dev.anom_fit)].reset_index(drop=True)
Xd, yd, gd = dv[FEATS].values, dv.anom_fit.values, dv.year.values


def tune5(name, X, y, g):
    gkf = GroupKFold(n_splits=5)
    scores = {}
    for i, p in enumerate(GRIDS[name]):
        scores[i] = np.mean([np.mean((make_model(name, p).fit(X[a], y[a]).predict(X[b]) - y[b]) ** 2)
                             for a, b in gkf.split(X, y, g)])
    return GRIDS[name][min(scores, key=scores.get)]


best = {m: tune5(m, Xd, yd, gd) for m in ("XGB", "RF", "Ridge")}
print("final hyperparameters:", best)
final = {m: make_model(m, best[m]).fit(Xd, yd) for m in ("XGB", "RF", "Ridge")}

B = 40
boot_path = CACHE / "boot_models.json"
seasons_dev = np.unique(gd)


def fit_boot(b):
    r = np.random.default_rng(SEED + b)
    s = r.choice(seasons_dev, size=len(seasons_dev), replace=True)
    ix = np.concatenate([np.where(gd == k)[0] for k in s])
    mx = make_model("XGB", best["XGB"], n_jobs=1, seed=SEED + b).fit(Xd[ix], yd[ix])
    mr = make_model("RF", best["RF"], n_jobs=1, seed=SEED + b).fit(Xd[ix], yd[ix])
    return mx, mr


t0 = time.time()
boots = Parallel(n_jobs=N_JOBS)(delayed(fit_boot)(b) for b in range(B))
print(f"{B} bootstrap refits of XGB and RF in {time.time() - t0:.0f} s")


def ens_predict(X, models=None):
    models = models or (final["XGB"], final["RF"])
    return np.mean([m.predict(X) for m in models], axis=0)


# supplementary model: climate + MODIS (2001-2025 seasons), LOYO
eo_dev = dv[dv.year >= 2001].dropna(subset=EO_FEATURES).reset_index(drop=True)
eo_rows = []
for fs_name, fs in [("climate only", FEATS), ("climate + MODIS", FEATS + EO_FEATURES)]:
    pr = np.zeros(len(eo_dev))
    for y in eo_dev.year.unique():
        a, b = eo_dev.year.values != y, eo_dev.year.values == y
        pr[b] = make_model("XGB", best["XGB"]).fit(eo_dev.loc[a, fs].values, eo_dev.anom_fit.values[a]).predict(
            eo_dev.loc[b, fs].values)
    e = pr - eo_dev.anom_fit.values
    eo_rows.append({"feature_set": fs_name, "n": len(eo_dev), "RMSE_anom": np.sqrt(np.mean(e ** 2)),
                    "R2_anom": r2_score(eo_dev.anom_fit, pr),
                    "SS_vs_trend": 1 - np.mean(e ** 2) / np.mean(eo_dev.anom_fit ** 2)})
eo_skill = pd.DataFrame(eo_rows)
eo_skill.round(4).to_csv(TAB / "T2c_eo_added_skill.csv", index=False)
print(eo_skill.round(3).to_string(index=False))

### 3.3 Feature-design experiment (Supplementary Table S3)
LOYO skill with (a) absolute predictors, (b) within-district anomalies of all predictors and (c) anomalies plus
absolute late-season heat, each with standard and strong regularisation. Absolute predictors were retained:
they performed at least as well and are required to read physical temperature thresholds.

In [ ]:
fe_path = CACHE / "feature_experiment.csv"
if not fe_path.exists():
    cols_c = [c for c in FEATS if not c.startswith("irr")]
    A_ = dev[cols_c] - dev.groupby("adm2_pcode")[cols_c].transform("mean")
    A_.columns = [c + "_a" for c in cols_c]
    dx = pd.concat([dev, A_], axis=1)
    late_abs = [f"{v}_{s}" for s in ("joint", "anth", "fill") for v in ("tmax", "kdd30", "kdd34")]
    SETS = {"absolute": FEATS, "anomaly": list(A_.columns) + ["irr_share"],
            "anomaly+late abs heat": list(A_.columns) + late_abs + ["irr_share"]}
    MK = {"XGB std": lambda: make_model("XGB", {"max_depth": 3, "min_child_weight": 20}),
          "XGB strong": lambda: xgb.XGBRegressor(n_estimators=400, learning_rate=0.02, max_depth=2, min_child_weight=60,
                                                 subsample=0.7, colsample_bytree=0.5, reg_lambda=10, n_jobs=N_JOBS,
                                                 random_state=1),
          "RF strong": lambda: make_model("RF", {"max_features": 0.33, "min_samples_leaf": 40})}
    rows_fe = []
    for sname, cols in SETS.items():
        for mname, mk in MK.items():
            o_, p_ = [], []
            for y in np.unique(yrs):
                an, _ = refit_anomaly(dx, yrs != y)
                ok = np.isfinite(an)
                m_ = mk().fit(dx.loc[(yrs != y) & ok, cols].values, an[(yrs != y) & ok])
                o_.append(an[(yrs == y) & ok]); p_.append(m_.predict(dx.loc[(yrs == y) & ok, cols].values))
            o_, p_ = np.concatenate(o_), np.concatenate(p_)
            rows_fe.append({"features": sname, "model": mname, "SS_vs_trend": 1 - np.mean((p_ - o_) ** 2) / np.mean(o_ ** 2),
                            "r": stats.pearsonr(o_, p_)[0]})
    pd.DataFrame(rows_fe).to_csv(fe_path, index=False)
fe = pd.read_csv(fe_path)
fe.round(4).to_csv(TAB / "S3_feature_design_experiment.csv", index=False)
print(fe.round(3).to_string(index=False))

## 4. SHAP attribution and stage-wise heat thresholds (RQ1)

TreeSHAP values of the final XGBoost (and, for robustness, the random forest) decompose each prediction.
For the stage mean maximum temperature the threshold is the temperature at which the LOWESS-smoothed SHAP
dependence crosses from positive to negative; its 95 % interval comes from 200 district-bootstrap samples.
The response above the threshold is the slope of SHAP on temperature (percentage points of yield per degC).
Accumulated local effects (ALE) of the XGB-RF ensemble give a model-agnostic check.

In [ ]:
import shap
from statsmodels.nonparametric.smoothers_lowess import lowess

expl_x = shap.TreeExplainer(final["XGB"])
sv_x = expl_x.shap_values(Xd)
r_sub = np.random.default_rng(SEED).choice(len(Xd), size=min(1500, len(Xd)), replace=False)
expl_r = shap.TreeExplainer(final["RF"])
sv_r = expl_r.shap_values(Xd[r_sub], check_additivity=False)

imp = pd.DataFrame({"feature": FEATS, "mean_abs_shap_XGB": np.abs(sv_x).mean(0),
                    "mean_abs_shap_RF": np.abs(sv_r).mean(0)}).sort_values("mean_abs_shap_XGB", ascending=False)


def fgroup(f):
    if f.startswith("irr"):
        return "irrigation", "-"
    v, s = f.split("_", 1)
    return v, s


imp["variable"], imp["stage"] = zip(*imp.feature.map(fgroup))
imp.round(5).to_csv(TAB / "T3a_shap_importance.csv", index=False)
print(imp.head(15).round(4).to_string(index=False))
print((imp.groupby("stage").mean_abs_shap_XGB.sum() / imp.mean_abs_shap_XGB.sum()).round(3).to_string())


def zero_cross(x, s, frac=0.4):
    """Lowest x at which the LOWESS curve of s on x turns negative after being positive."""
    lo = lowess(s, x, frac=frac, return_sorted=True)
    xs, ys = lo[:, 0], lo[:, 1]
    for i in range(1, len(xs)):
        if ys[i - 1] > 0 >= ys[i]:
            return xs[i - 1] + (xs[i] - xs[i - 1]) * ys[i - 1] / (ys[i - 1] - ys[i]), lo
    return np.nan, lo


def threshold_table(sv, X, rows_idx, label):
    out = []
    dists = dv.adm2_pcode.values[rows_idx]
    ud = np.unique(dists)
    rb = np.random.default_rng(SEED)
    for s in STAGE_ORDER:
        j = FEATS.index(f"tmax_{s}")
        x, sh = X[:, j], sv[:, j]
        thr, lo = zero_cross(x, sh)
        above = x > thr if np.isfinite(thr) else np.zeros(len(x), bool)
        slope = stats.theilslopes(sh[above], x[above]).slope * 100 if above.sum() > 20 else np.nan
        bt = []
        for _ in range(200):
            pick = rb.choice(ud, size=len(ud), replace=True)
            ix = np.concatenate([np.where(dists == d)[0] for d in pick])
            bt.append(zero_cross(x[ix], sh[ix])[0])
        bt = np.array(bt)
        out.append({"model": label, "stage": s, "threshold_C": thr, "ci_lo": np.nanpercentile(bt, 2.5),
                    "ci_hi": np.nanpercentile(bt, 97.5), "share_boot_crossing": np.isfinite(bt).mean(),
                    "slope_pp_per_C_above": slope, "pct_obs_above": 100 * above.mean(),
                    "tmax_p5": np.percentile(x, 5), "tmax_p95": np.percentile(x, 95)})
    return pd.DataFrame(out)


thr_x = threshold_table(sv_x, Xd, np.arange(len(Xd)), "XGB")
thr_r = threshold_table(sv_r, Xd[r_sub], r_sub, "RF")


def ale_1d(predict, X, j, nbins=20):
    x = X[:, j]
    edges = np.unique(np.quantile(x, np.linspace(0, 1, nbins + 1)))
    k = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, len(edges) - 2)
    eff = np.zeros(len(edges) - 1)
    cnt = np.zeros(len(edges) - 1)
    lo_, hi_ = X.copy(), X.copy()
    lo_[:, j], hi_[:, j] = edges[k], edges[k + 1]
    d = predict(hi_) - predict(lo_)
    for b in range(len(edges) - 1):
        cnt[b] = (k == b).sum()
        eff[b] = d[k == b].mean() if cnt[b] else 0
    ale = np.concatenate([[0], np.cumsum(eff)])
    centers = ale - np.sum((ale[:-1] + ale[1:]) / 2 * cnt) / cnt.sum()
    return edges, centers


ale_res, ale_thr = {}, []
for s in STAGE_ORDER:
    j = FEATS.index(f"tmax_{s}")
    e, a = ale_1d(ens_predict, Xd, j)
    ale_res[s] = (e, a)
    cross = [e[i - 1] + (e[i] - e[i - 1]) * a[i - 1] / (a[i - 1] - a[i]) for i in range(1, len(e))
             if a[i - 1] > 0 >= a[i]]
    ale_thr.append({"stage": s, "ALE_threshold_C": cross[0] if cross else np.nan,
                    "ALE_range_pp": 100 * (a.max() - a.min())})
thr = pd.concat([thr_x, thr_r]).merge(pd.DataFrame(ale_thr), on="stage", how="left")
thr.round(3).to_csv(TAB / "T3_heat_thresholds.csv", index=False)
print(thr.round(2).to_string(index=False))

## 5. Heat-attributable losses and hazard

For every district and season the ensemble prediction under observed conditions is compared with the
prediction in which the heat predictors (stage Tmax, KDD30, KDD34 and the irrigation-heat interaction) are
set to the district's 1991-2020 climatology (Eq. 3). The hazard H_d is the mean loss over 1991-2020,
expressed as a fraction of trend yield; it is also converted to tonnes with the 2025 trend yield and the
2021-25 mean harvested area.

In [ ]:
risk_d = sorted(dv.adm2_pcode.unique())
base = feat[feat.adm2_pcode.isin(risk_d) & feat.year.between(1991, 2020)].reset_index(drop=True)
clim_heat = base.groupby("adm2_pcode")[HEAT_VARS].transform("mean")


def heat_loss(frame, clim_frame, models=None, heat_vars=None):
    heat_vars = heat_vars or HEAT_VARS
    Xo = frame[FEATS].copy()
    Xc = Xo.copy()
    Xc[heat_vars] = clim_frame[heat_vars].values
    Xc["irr_x_kdd30_late"] = Xc.irr_share * (Xc.kdd30_anth + Xc.kdd30_fill)
    fo, fc = ens_predict(Xo.values, models), ens_predict(Xc.values, models)
    return np.maximum(0.0, fc - fo), fo, fc


L, f_obs, f_clim = heat_loss(base, clim_heat)
base["loss"] = L
hz = base.groupby("adm2_pcode").loss.agg(H="mean", H_p90=lambda v: np.percentile(v, 90)).reset_index()
H_boot = np.vstack([base.assign(l=heat_loss(base, clim_heat, models=m)[0]).groupby("adm2_pcode").l.mean()
                    .reindex(risk_d).values for m in boots])  # B x districts

# convert to tonnes: 2025 trend yield x 2021-25 mean area
ty = tgt.groupby("adm2_pcode").apply(lambda g: g.trend_t_ha[g.year == g.year.max()].iloc[0]
                                     + g.slope_t_ha_yr.iloc[0] * (2025 - g.year.max()))
area_recent = panel[panel.own & panel.year.between(2021, 2025)].groupby("adm2_pcode").area_ha.mean()
area_any = panel[panel.own].groupby("adm2_pcode").area_ha.apply(lambda v: v.tail(5).mean())
hz["trend2025_t_ha"] = hz.adm2_pcode.map(ty)
hz["area_ha"] = hz.adm2_pcode.map(area_recent).fillna(hz.adm2_pcode.map(area_any))
hz["H_t"] = hz.H * hz.trend2025_t_ha * hz.area_ha
hz = hz.merge(study[["adm2_pcode", "adm2_name", "adm1_name"]], on="adm2_pcode")
yr_loss = base.groupby("year").apply(lambda g: np.average(g.loss, weights=g.adm2_pcode.map(hz.set_index("adm2_pcode").area_ha)))
print("national area-weighted mean heat loss 1991-2020 (% of trend yield):", round(100 * yr_loss.mean(), 2))
print("worst seasons:", (100 * yr_loss.sort_values(ascending=False).head(5)).round(2).to_dict())
print("expected annual heat loss (kt):", round(hz.H_t.sum() / 1e3, 1))
print(hz.sort_values("H", ascending=False).head(10)[["adm2_name", "adm1_name", "H", "H_t"]].round(4).to_string(index=False))

### 5.1 Future hazard at 1.5, 2 and 3 degC global warming (NEX-GDDP-CMIP6, delta method)

For each NEX-GDDP-CMIP6 model, the 20-year window in which it reaches a global warming level (GWL; relative
to 1850-1900, Hauser et al. warming-level table, SSP5-8.5, SSP2-4.5 where SSP5-8.5 is unavailable) is
compared with its own 1991-2020 climate. Monthly district changes in Tmax and Tmin (additive) and rainfall
(multiplicative, bounded to 0.2-5) are applied to the observed 1991-2020 daily series; dew point is shifted
with mean temperature (approximately constant relative humidity) and soil moisture is held at observed
values. Predictors are recomputed and the heat loss is evaluated against the observed 1991-2020 heat
climatology. CO2 fertilisation and adaptation are not represented.

In [ ]:
NEX = CLIM / "nex_gddp_cmip6"
GWLS = [1.5, 2.0, 3.0]
wl = pd.read_csv(CLIM / "cmip6_warming_levels" / "cmip6_warming_levels_one_ens_1850_1900.csv", comment="#",
                 skipinitialspace=True)
wl.columns = [c.strip() for c in wl.columns]
gcms_all = sorted(os.path.basename(p) for p in glob.glob(str(NEX / "*")) if os.path.isdir(p))
# daily Tmax and Tmin are required; CESM2, CESM2-WACCM and IITM-ESM provide only pr and tas in NEX-GDDP
gcms = [m for m in gcms_all if glob.glob(str(NEX / m / "historical" / "tasmax_*"))
        and not m.endswith("_gr2")]  # GFDL-CM4_gr2 is GFDL-CM4 on a second grid: counted once
print(f"NEX-GDDP models: {len(gcms_all)} on disk, {len(gcms)} used (daily Tmax/Tmin, one grid per model):",
      sorted(set(gcms_all) - set(gcms)), "excluded")


def gwl_windows(model):
    name = model.replace("_gr2", "")
    out = {}
    for g in GWLS:
        for exp in ("ssp585", "ssp245"):
            r = wl[(wl.model == name) & (wl.exp == exp) & np.isclose(wl.warming_level, g)]
            if len(r) and os.path.isdir(NEX / model / exp) and len(os.listdir(NEX / model / exp)):
                out[g] = (exp, int(r.start_year.iloc[0]), int(r.end_year.iloc[0]))
                break
    return out


PTS = (pts.lat.values, pts.lon.values, pts.w.values, pts.d.values)
MONTHS = [11, 12, 1, 2, 3, 4]


def nex_monthly(model):
    """Monthly (Nov-Apr) wheat-weighted district means of tasmax, tasmin, pr for the needed years."""
    out = CACHE / "nex" / f"{model}.parquet"
    if out.exists():
        return pd.read_parquet(out)
    out.parent.mkdir(exist_ok=True)
    wins = gwl_windows(model)
    fut_exp = "ssp585" if os.path.isdir(NEX / model / "ssp585") and len(os.listdir(NEX / model / "ssp585")) else "ssp245"
    need = {(y, "historical") for y in range(1991, 2015)} | {(y, fut_exp) for y in range(2015, 2021)}
    for g, (exp, a, b) in wins.items():
        need |= {(y, exp) for y in range(a, b + 1)}
    plat, plon, pw, pd_ = PTS
    W, rec = None, []
    for (y, exp) in sorted(need):
        for var in ("tasmax", "tasmin", "pr"):
            fs = (glob.glob(str(NEX / model / exp / f"{var}_day_*_{exp}_*_{y}_v*.nc"))
                  + glob.glob(str(NEX / model / exp / f"{var}_day_*_{exp}_*_{y}.nc")))
            if not fs:
                continue
            with xr.open_dataset(fs[0], engine="h5netcdf") as ds:
                if W is None:
                    lat, lon = ds.lat.values, ds.lon.values
                    il = np.abs(plat[:, None] - lat[None, :]).argmin(1)
                    jl = np.abs(plon[:, None] - lon[None, :]).argmin(1)
                    W = sparse.csr_matrix((pw, (pd_, il * len(lon) + jl)), shape=(len(PCODES), len(lat) * len(lon)))
                mo = ds.time.dt.month.values
                arr = ds[var].transpose("time", "lat", "lon").values.astype("float64")
            dm = district_mean(W, arr)
            for m in MONTHS:
                sel = mo == m
                if sel.any():
                    rec.append(pd.DataFrame({"model": model, "exp": exp, "var": var, "year": y, "month": m,
                                             "adm2_pcode": PCODES, "value": dm[sel].mean(0)}))
    df = pd.concat(rec, ignore_index=True) if rec else pd.DataFrame(
        columns=["model", "exp", "var", "year", "month", "adm2_pcode", "value"])
    df.to_parquet(out)
    return df


t0 = time.time()
nex_all = Parallel(n_jobs=6)(delayed(nex_monthly)(m) for m in gcms)
print(f"NEX-GDDP monthly district means for {len(gcms)} models in {(time.time() - t0) / 60:.1f} min")

delta_rows, used = [], []
for model, df in zip(gcms, nex_all):
    wins = gwl_windows(model)
    piv = df.pivot_table(index=["year", "month", "adm2_pcode"], columns="var", values="value").reset_index()
    if not {"tasmax", "tasmin", "pr"} <= set(piv.columns):
        continue
    basec = piv[piv.year.between(1991, 2020)].groupby(["month", "adm2_pcode"])[["tasmax", "tasmin", "pr"]].mean()
    for g, (exp, a, b) in wins.items():
        futc = piv[piv.year.between(a, b)].groupby(["month", "adm2_pcode"])[["tasmax", "tasmin", "pr"]].mean()
        d = pd.DataFrame({"dtmax": futc.tasmax - basec.tasmax, "dtmin": futc.tasmin - basec.tasmin,
                          "pr_ratio": (futc.pr / basec.pr.replace(0, np.nan)).clip(0.2, 5).fillna(1.0)}).reset_index()
        d["model"], d["gwl"], d["exp"], d["window"] = model, g, exp, f"{a}-{b}"
        delta_rows.append(d)
        used.append({"model": model, "gwl": g, "exp": exp, "start": a, "end": b})
deltas = pd.concat(delta_rows, ignore_index=True)
used = pd.DataFrame(used)
used.to_csv(TAB / "S1_gcm_warming_windows.csv", index=False)
print(used.groupby("gwl").model.nunique().rename("n_models").to_string())
w_area = hz.set_index("adm2_pcode").area_ha
dsum = deltas[deltas.adm2_pcode.isin(risk_d)].groupby(["gwl", "month"]).apply(
    lambda g: pd.Series({"dTmax_C": np.average(g.dtmax, weights=g.adm2_pcode.map(w_area)),
                         "rain_ratio": np.average(g.pr_ratio, weights=g.adm2_pcode.map(w_area))}))
print("wheat-area-weighted multi-model mean change:\n", dsum.groupby(level=[0, 1]).mean().round(2).unstack(0).to_string())

In [ ]:
obs_win = (tmax.index >= "1990-11-01") & (tmax.index <= "2020-04-30")
TX, TN = tmax[obs_win], tmin[obs_win]
TD = dew.reindex(TX.index)
PR = prcp.reindex(TX.index)
SM = sm.reindex(TX.index)
mon = TX.index.month.values
SEAS_BASE = list(range(1991, 2021))


def future_features(d):
    dx = d.pivot(index="month", columns="adm2_pcode", values="dtmax").reindex(columns=PCODES).fillna(0)
    dn = d.pivot(index="month", columns="adm2_pcode", values="dtmin").reindex(columns=PCODES).fillna(0)
    rr = d.pivot(index="month", columns="adm2_pcode", values="pr_ratio").reindex(columns=PCODES).fillna(1)
    ax = dx.reindex(mon).fillna(0).values
    an = dn.reindex(mon).fillna(0).values
    ar = rr.reindex(mon).fillna(1).values
    f = stage_features(TX + ax, TN + an, TD + (ax + an) / 2, PR * ar, SM, SEAS_BASE)
    f = f.merge(irr_share, left_on="adm2_pcode", right_index=True, how="left")
    f["irr_x_kdd30_late"] = f.irr_share * (f.kdd30_anth + f.kdd30_fill)
    return f[f.adm2_pcode.isin(risk_d)].sort_values(["adm2_pcode", "year"]).reset_index(drop=True)


base_sorted = base.sort_values(["adm2_pcode", "year"]).reset_index(drop=True)
clim_sorted = base_sorted.groupby("adm2_pcode")[HEAT_VARS].transform("mean")
fut_path = CACHE / "future_hazard.parquet"
if fut_path.exists():
    fut = pd.read_parquet(fut_path)
else:
    def one(model, g):
        d = deltas[(deltas.model == model) & (deltas.gwl == g)]
        ff = future_features(d)
        l, fo, _ = heat_loss(ff, clim_sorted)
        r = ff[["adm2_pcode", "year"]].copy()
        r["loss"], r["model"], r["gwl"] = l, model, g
        r["tmax_fill"], r["kdd34_fill"], r["kdd34_anth"] = ff.tmax_fill.values, ff.kdd34_fill.values, ff.kdd34_anth.values
        return r
    t0 = time.time()
    fut = pd.concat([one(r.model, r.gwl) for r in used.itertuples()], ignore_index=True)
    fut.to_parquet(fut_path)
    print(f"future losses for {len(used)} model-GWL pairs in {(time.time() - t0) / 60:.1f} min")

Hf = fut.groupby(["gwl", "model", "adm2_pcode"]).loss.mean().rename("H").reset_index()
Hf_med = Hf.groupby(["gwl", "adm2_pcode"]).H.median().unstack(0)
hz = hz.set_index("adm2_pcode")
for g in GWLS:
    hz[f"H_{g}"] = Hf_med[g].reindex(hz.index)
    hz[f"H_t_{g}"] = hz[f"H_{g}"] * hz.trend2025_t_ha * hz.area_ha
hz = hz.reset_index()
nat = {"present": np.average(hz.H, weights=hz.area_ha)}
for g in GWLS:
    nat[f"+{g}C"] = np.average(hz[f"H_{g}"], weights=hz.area_ha)
gcm_spread = Hf.groupby(["gwl", "model"]).apply(lambda g: np.average(g.H, weights=g.adm2_pcode.map(w_area)))
fut_tab = pd.DataFrame({
    "period": ["1991-2020"] + [f"+{g} degC" for g in GWLS],
    "n_models": [np.nan] + [used[used.gwl == g].model.nunique() for g in GWLS],
    "loss_pct_trend_yield": [100 * nat["present"]] + [100 * nat[f"+{g}C"] for g in GWLS],
    "gcm_p10": [np.nan] + [100 * np.percentile(gcm_spread[g], 10) for g in GWLS],
    "gcm_p90": [np.nan] + [100 * np.percentile(gcm_spread[g], 90) for g in GWLS],
    "loss_kt": [hz.H_t.sum() / 1e3] + [hz[f"H_t_{g}"].sum() / 1e3 for g in GWLS],
})
fut_tab.round(3).to_csv(TAB / "T4_hazard_warming_levels.csv", index=False)
print(fut_tab.round(2).to_string(index=False))
hz.to_csv(TAB / "district_hazard.csv", index=False)

## 6. Exposure (Stage B) and vulnerability (Stage C)

District socioeconomic indicators are read from the original statistical tables:

* Agricultural Census 2024 (PBS), district tables: T1 farms by tenure, T3 farms by size, T4 cultivated area by
  mode of irrigation, T5 share of wheat in cropped area;
* Population and Housing Census 2023 (PBS): T1 rural population and household size, T12 rural literacy,
  T22 lighting fuel, T24 toilet facilities;
* Multidimensional Poverty in Pakistan (2016): district MPI headcount and intensity, 2014-15;
* travel time to the nearest city (Weiss et al. 2018), Relative Wealth Index (Chi et al. 2022) and VIIRS
  night-time lights (district means computed in Google Earth Engine).

Names are matched to the 2022 OCHA districts; districts created after a survey inherit the value of their
parent district.

In [ ]:
SOC = RAW / "05_socioeconomic"
ADM_KEY = {re.sub(r"[^A-Z]", "", n.upper()): pc for n, pc in zip(study.adm2_name, study.adm2_pcode)}
ALIAS_SOC = {"LAYYAH": "Leiah", "SURAB": "Shaheed Sikandarabad", "KARACHIEAST": "East Karachi",
             "KARACHIWEST": "West Karachi", "KEAMARI": "West Karachi", "KARACHICENTRAL": "Central Karachi",
             "KARACHISOUTH": "South Karachi", "KORANGI": "Korangi Karachi", "MALIR": "Malir Karachi",
             "TORGHAR": "Tor Ghar", "DERAISMAILKHAN": "D. I. Khan", "DIKHAN": "D. I. Khan", "TANDOAHYAR": "Tando Allahyar",
             "TANDOALLAHYAR": "Tando Allahyar", "UPPERCHITRAL": "Chitral Upper", "LOWERCHITRAL": "Chitral Lower",
             "UPPERKOHISTAN": "Kohistan Upper", "LOWERKOHISTAN": "Kohistan Lower",
             "SHAHEEDBENAZIRABAD": "Shaheed Benazir Abad", "NAWABSHAHSHAHEEDBENAZIRABAD": "Shaheed Benazir Abad",
             "BOLANKACHHI": "Kachhi", "KECHTURBAT": "Kech", "NAUSHEHROFEROZE": "Naushahro Feroze",
             "NOWSHEHRA": "Nowshera", "GAWADAR": "Gwadar", "DGKHAN": "Dera Ghazi Khan", "MIRPURKHAS": "Mirpur Khas",
             "UMERKOT": "Umer Kot", "TTSINGH": "Toba Tek Singh", "KAMBARSHAHDADKOT": "Kambar Shahdad Kot",
             "BATAGRAM": "Batagram", "MUSAKHEL": "Musakhel", "KILLAABDULLAH": "Killa Abdullah",
             "TANDOMUHAMMADKHAN": "Tando Muhammad Khan", "KOHISTAN": "Kohistan Upper", "CHITRAL": "Chitral Upper",
             "LEHRI": "Lehri", "SHERANI": "Sherani", "MALAKANDPROTECTEDAREA": "Malakand",
             "SARGHODHA": "Sargodha", "TORGARH": "Tor Ghar", "KARACHI": "South Karachi"}
NAME2PC = {n: pc for n, pc in zip(study.adm2_name, study.adm2_pcode)}


def to_pcode(name):
    k = re.sub(r"\s*DISTRICT\s*$", "", " ".join(str(name).split()).upper())
    k = re.sub(r"[^A-Z]", "", k)
    if k in ALIAS_SOC:
        return NAME2PC.get(ALIAS_SOC[k])
    return ADM_KEY.get(k)


# parent fill for districts created after a survey (same lineage table as the yield harmonisation)
PARENT_SOC = {"Chaman": "Killa Abdullah", "Duki": "Loralai", "Shaheed Sikandarabad": "Kalat", "Sohbatpur": "Jaffarabad",
              "Lehri": "Sibi", "Kohistan Lower": "Kohistan Upper", "Kolai Palas Kohistan": "Kohistan Upper",
              "Chitral Lower": "Chitral Upper", "Tor Ghar": "Mansehra", "Sujawal": "Thatta", "Harnai": "Sibi",
              "Washuk": "Kharan", "Nushki": "Chagai", "Sherani": "Zhob", "Chiniot": "Jhang", "Kashmore": "Jacobabad",
              "Kambar Shahdad Kot": "Larkana", "Jamshoro": "Dadu", "Matiari": "Hyderabad", "Tando Allahyar": "Hyderabad",
              "Tando Muhammad Khan": "Hyderabad", "Nankana Sahib": "Sheikhupura"}


def fill_parent(s):
    s = s.copy()
    for child, par in PARENT_SOC.items():
        c, p = NAME2PC.get(child), NAME2PC.get(par)
        if c and p and pd.isna(s.get(c, np.nan)) and pd.notna(s.get(p, np.nan)):
            s[c] = s[p]
    return s


def parse_blocks(path, label_col=0, header_pat=r"DISTRICT\s*$"):
    """Yield (pcode, block DataFrame) for each 'X DISTRICT' block of a PBS district table."""
    d = pd.read_excel(path, header=None)
    lab = d[label_col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    starts = [i for i, v in enumerate(lab) if re.search(header_pat, v.upper()) and "DIVISION" not in v.upper()]
    for a, b in zip(starts, starts[1:] + [len(d)]):
        pc = to_pcode(lab[a])
        if pc:
            blk = d.iloc[a + 1:b].copy()
            blk.index = lab[a + 1:b].str.upper()
            yield pc, blk


def first_row(blk, key):
    r = blk[blk.index.str.startswith(key)]
    if not len(r):
        return None
    return pd.to_numeric(r.iloc[0].astype(str).str.replace(",", "").str.strip(), errors="coerce")


AG = SOC / "agricultural_census_2024"
agc = {}
for pc, blk in parse_blocks(AG / "Table_1_District.xls"):
    r = first_row(blk, "PRIVATE-FARMS TOTAL")
    if r is not None:
        agc.setdefault(pc, {}).update(farms=r[3], tenant_share=(r[7] + r[9]) / r[3] if r[3] else np.nan)
for pc, blk in parse_blocks(AG / "Table_3_District.xls"):
    tot = first_row(blk, "PRIVATE-FARMS TOTAL")
    small = sum(first_row(blk, k)[1] for k in ("UNDER 1.0", "1.0 TO UNDER 2.5", "2.5 TO UNDER 5.0")
                if first_row(blk, k) is not None)
    if tot is not None:
        agc.setdefault(pc, {}).update(small_farm_share=small / tot[1], cultivated_acres=tot[5])
for pc, blk in parse_blocks(AG / "Table_4_District.xls"):
    r = first_row(blk, "PRIVATE-FARMS TOTAL")
    if r is not None and r[2]:
        agc.setdefault(pc, {}).update(rainfed_share=r[14] / r[2])
for pc, blk in parse_blocks(AG / "Table_5_District.xls"):
    r = first_row(blk, "PRIVATE-FARMS TOTAL")
    if r is not None:
        agc.setdefault(pc, {}).update(cropped_acres=r[1], wheat_share_cropped=pd.to_numeric(r[2], errors="coerce") / 100)
agc = pd.DataFrame(agc).T.apply(pd.to_numeric, errors="coerce")
print("Agricultural Census 2024 districts matched:", len(agc))

CT = SOC / "population_census2023" / "census2023_detailed_tables_xlsx"


def census_files(t):
    fs = sorted(glob.glob(str(CT / f"table_{t}_*_district*.xlsx"))) + [str(CT / f"table_{t}_islamabad.xlsx")]
    return [f for f in fs if os.path.exists(f)]


cen = {}
for f in census_files(1):
    for pc, blk in parse_blocks(f):
        r = first_row(blk, "RURAL")
        if r is not None:
            cen.setdefault(pc, {}).update(rural_pop=r[2], rural_hh=(r[2] / r[9]) if r[2] > 0 else 0.0)
for f in census_files(12):
    for pc, blk in parse_blocks(f):
        r = first_row(blk, "LITERATE %")
        if r is not None:
            # districts without rural population (Lahore) use the total literacy rate
            cen.setdefault(pc, {}).update(rural_literacy=(r[5] if r[5] > 0 else r[1]) / 100)
for f in census_files(22):
    for pc, blk in parse_blocks(f):
        r = first_row(blk, "RURAL")
        if r is not None and r[1]:
            cen.setdefault(pc, {}).update(no_electricity=1 - r[2] / r[1], firewood=r[7] / r[1])
for f in census_files(24):
    for pc, blk in parse_blocks(f):
        r = first_row(blk, "RURAL")
        if r is not None and r[1]:
            cen.setdefault(pc, {}).update(no_toilet=r[5] / r[1])
cen = pd.DataFrame(cen).T.apply(pd.to_numeric, errors="coerce")
print("Census 2023 districts matched:", len(cen))

# MPI 2016 statistical annex (district table, PDF pages 44-45)
from pypdf import PdfReader

mpi_rows = []
rd = PdfReader(str(SOC / "mpi_2016" / "Multidimensional_Poverty_in_Pakistan_2016.pdf"))
TOKEN = r"(?:\d+\.\d+%?|\*)"
for pg in (43, 44):
    lines = (rd.pages[pg].extract_text() or "").split("\n")
    dl = [m for m in (re.match(rf"^([A-Za-z][A-Za-z .'/()\-]+?)\s+((?:{TOKEN}\s+){{7}}{TOKEN})\s*$", l.strip())
                      for l in lines) if m and not m.group(1).startswith("Districts")]
    tl = [l.split() for l in lines if re.fullmatch(rf"(?:{TOKEN}\s*){{10}}", l.strip() + " ")]
    assert len(dl) == len(tl), (pg, len(dl), len(tl))
    for m, t in zip(dl, tl):
        v = m.group(2).split()
        num = lambda x: np.nan if x == "*" else float(x.rstrip("%")) / (100 if x.endswith("%") else 1)
        H = next((num(x) for x in (t[3], t[2], t[1]) if x != "*"), np.nan)
        A = next((num(x) for x in (t[9], t[8], t[7]) if x != "*"), np.nan)
        mpi_rows.append({"name": m.group(1).strip(), "pcode": to_pcode(m.group(1)), "mpi": num(v[5]) if v[5] != "*" else num(v[4]),
                         "mpi_headcount": H, "mpi_intensity": A})
mpi = pd.DataFrame(mpi_rows)
print("MPI districts parsed:", len(mpi), "| unmatched:", mpi[mpi.pcode.isna()].name.tolist())
mpi = mpi.dropna(subset=["pcode"]).groupby("pcode")[["mpi", "mpi_headcount", "mpi_intensity"]].mean()
for k in ("Central Karachi", "East Karachi", "Korangi Karachi", "Malir Karachi", "West Karachi"):  # one Karachi value
    mpi.loc[NAME2PC[k]] = mpi.loc[NAME2PC["South Karachi"]]

EO = RAW / "03_earth_observation"
tt = pd.read_csv(EO / "travel_time_cities_2015" / "travel_time_district.csv").set_index("adm2_pcode").traveltime_mean_min
vi = pd.read_csv(EO / "viirs_nightlights" / "viirs_dnb_annual_district_2013_2025.csv")
vi = vi[vi.year.between(2021, 2023)].groupby("adm2_pcode").radiance_mean_nW_cm2_sr.mean()
rwi = pd.read_csv(SOC / "relative_wealth_index" / "pak_relative_wealth_index_points_with_district.csv")
rwi = rwi.groupby("adm2_pcode").rwi.mean()

soc = pd.DataFrame(index=PCODES)
soc = soc.join(agc).join(cen).join(mpi)
soc["travel_time_min"], soc["viirs_rad"], soc["rwi"] = tt, vi, rwi
for c in soc.columns:
    soc[c] = fill_parent(soc[c])
soc["housing_deprivation"] = soc[["no_electricity", "no_toilet"]].mean(axis=1)
soc.index.name = "adm2_pcode"
soc.to_csv(TAB / "district_socioeconomic_indicators.csv")
print(soc.loc[risk_d].describe().T[["count", "mean", "min", "max"]].round(3).to_string())

### 6.1 Exposure
E_d combines the share of wheat in the cropped area (Agricultural Census 2024) and the number of rural
households (Census 2023, log(1+x) scale), each min-max normalised across the districts of the risk analysis.

In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())


R = soc.loc[risk_d].copy()
prov_of = study.set_index("adm2_pcode").adm1_name
for c in R.columns:  # remaining gaps: provincial median
    R[c] = R[c].fillna(R.groupby(prov_of.reindex(R.index))[c].transform("median")).fillna(R[c].median())
R["E"] = (minmax(R.wheat_share_cropped) + minmax(np.log1p(R.rural_hh))) / 2

### 6.2 Vulnerability index (Eq. 4)
Eight sensitivity and adaptive-capacity indicators are standardised and oriented so that higher means more
vulnerable. Weights come from a principal component analysis: for every component retained by the Kaiser
criterion (eigenvalue > 1) the squared loadings, scaled to sum to one, are weighted by the component's share
of retained variance and summed. Equal weights are a sensitivity test, and the index is checked against the
Relative Wealth Index and night-time light radiance (convergent validity).

In [ ]:
VULN = {"mpi_headcount": +1, "mpi_intensity": +1, "small_farm_share": +1, "rainfed_share": +1,
        "tenant_share": +1, "rural_literacy": -1, "housing_deprivation": +1, "travel_time_min": +1}
VLAB = {"mpi_headcount": "Poverty headcount", "mpi_intensity": "Poverty intensity", "small_farm_share": "Farms < 5 acres",
        "rainfed_share": "Rainfed cultivated area", "tenant_share": "Tenant-operated farms",
        "rural_literacy": "Rural illiteracy", "housing_deprivation": "Housing deprivation",
        "travel_time_min": "Travel time to city"}
Z = pd.DataFrame({k: s * (R[k] - R[k].mean()) / R[k].std() for k, s in VULN.items()})
C = np.corrcoef(Z.values.T)
eigval, eigvec = np.linalg.eigh(C)
order = np.argsort(eigval)[::-1]
eigval, eigvec = eigval[order], eigvec[:, order]
keep_c = eigval > 1
load = eigvec[:, keep_c] * np.sqrt(eigval[keep_c])
sq = load ** 2 / (load ** 2).sum(0)
w_pca = (sq * (eigval[keep_c] / eigval[keep_c].sum())).sum(1)
w_pca = pd.Series(w_pca / w_pca.sum(), index=list(VULN))
R["V_raw"] = Z.values @ w_pca.values
R["V_eq"] = Z.mean(axis=1)
R["V"] = minmax(R.V_raw)
pca_tab = pd.DataFrame({"indicator": [VLAB[k] for k in VULN], "orientation": list(VULN.values()),
                        "weight_PCA": w_pca.values, "weight_equal": 1 / len(VULN),
                        **{f"loading_PC{i + 1}": load[:, i] for i in range(load.shape[1])}})
pca_tab.round(4).to_csv(TAB / "T5_vulnerability_weights.csv", index=False)
print("eigenvalues:", eigval.round(3), "| retained:", keep_c.sum(),
      f"| variance explained: {100 * eigval[keep_c].sum() / eigval.sum():.1f} %")
print(pca_tab.round(3).to_string(index=False))

valid = {
    "Spearman V_PCA vs V_equal": stats.spearmanr(R.V_raw, R.V_eq),
    "Spearman V vs Relative Wealth Index": stats.spearmanr(R.V_raw, R.rwi, nan_policy="omit"),
    "Spearman V vs log night-light radiance": stats.spearmanr(R.V_raw, np.log(R.viirs_rad + 0.01), nan_policy="omit"),
    "Spearman V vs MPI (2014-15)": stats.spearmanr(R.V_raw, R.mpi, nan_policy="omit"),
}
valid_tab = pd.DataFrame([{"test": k, "rho": v[0], "p_value": v[1], "n": len(R)} for k, v in valid.items()])
valid_tab.round(4).to_csv(TAB / "T5b_vulnerability_validity.csv", index=False)
print(valid_tab.round(3).to_string(index=False))

## 7. Livelihood risk with Monte Carlo uncertainty (Stage D)

R_d = H~_d * E~_d * V~_d (Eq. 5). Hazard is rescaled with the present-climate range so that future values can
exceed 1. Each of the 10,000 draws samples (i) one bootstrap refit of the hazard model, (ii) vulnerability
weights from a Dirichlet distribution centred on the PCA weights (concentration 200), and (iii) for warming
levels, one CMIP6 model (its hazard scaled by the bootstrap-to-central ratio of the draw).

In [ ]:
NDRAW = 10_000
rng = np.random.default_rng(SEED)
R = R.loc[risk_d]
hzi = hz.set_index("adm2_pcode").loc[risk_d]
H0 = hzi.H.values
hmin, hmax = H0.min(), H0.max()
Hs = lambda h: (h - hmin) / (hmax - hmin)
Et = R.E.values
Zv = Z.loc[risk_d].values
ratio = np.where(H0 > 0, H_boot / np.where(H0 > 0, H0, 1), 1.0)  # B x d
Hf_arr = {g: Hf[Hf.gwl == g].pivot(index="model", columns="adm2_pcode", values="H")[risk_d].values for g in GWLS}


def vt_draw(w):
    v = Zv @ w
    return (v - v.min()) / (v.max() - v.min())


def mc(period):
    out = np.empty((NDRAW, len(risk_d)))
    for i in range(NDRAW):
        b = rng.integers(len(boots))
        w = rng.dirichlet(w_pca.values * 200)
        h = H_boot[b] if period == "present" else Hf_arr[period][rng.integers(len(Hf_arr[period]))] * ratio[b]
        out[i] = Hs(h) * Et * vt_draw(w)
    return out


mc_res = {p: mc(p) for p in ["present"] + GWLS}
riskdf = pd.DataFrame(index=risk_d)
for p, arr in mc_res.items():
    tag = "now" if p == "present" else f"{p}"
    rk = (-arr).argsort(1).argsort(1) + 1  # rank 1 = highest risk
    riskdf[f"R_{tag}"] = np.median(arr, 0)
    riskdf[f"R_{tag}_p05"], riskdf[f"R_{tag}_p95"] = np.percentile(arr, 5, 0), np.percentile(arr, 95, 0)
    riskdf[f"rank_{tag}"] = pd.Series(np.median(arr, 0), index=risk_d).rank(ascending=False).values
    riskdf[f"rank_{tag}_lo"], riskdf[f"rank_{tag}_hi"] = np.percentile(rk, 5, 0), np.percentile(rk, 95, 0)
riskdf["H_t"], riskdf["E_t"], riskdf["V_t"] = Hs(H0), Et, R.V.values
riskdf["R_point"] = riskdf.H_t * riskdf.E_t * riskdf.V_t
riskdf = riskdf.join(study.set_index("adm2_pcode")[["adm2_name", "adm1_name"]])
riskdf.index.name = "adm2_pcode"
riskdf.round(5).to_csv(TAB / "district_risk.csv")
print("rank interval width (median):", np.median(riskdf.rank_now_hi - riskdf.rank_now_lo))
print(riskdf.sort_values("R_now", ascending=False).head(15)[
    ["adm2_name", "adm1_name", "R_now", "rank_now_lo", "rank_now_hi", "H_t", "E_t", "V_t"]].round(3).to_string())
print("risk by province (median of district medians):\n", riskdf.groupby("adm1_name")[["R_now", "R_2.0", "R_3.0"]].median().round(3))

## 8. Hotspots and inequality (Stage E)

Local Moran's I (Eq. 6) of median present risk with queen contiguity among the risk districts, 9,999
conditional permutations and Benjamini-Hochberg false-discovery-rate control (q = 0.05). Inequality of heat
losses across the wealth distribution is measured with the concentration index (Eq. 7), districts ranked by
the mean Relative Wealth Index (poorest first, so C < 0 means losses concentrate among poorer districts), and
with the share of losses borne by the poorest 20 % of districts.

In [ ]:
import esda
from libpysal.weights import Queen

gd_risk = study.set_index("adm2_pcode").loc[risk_d].reset_index()
wq = Queen.from_dataframe(gd_risk, use_index=False, silence_warnings=True)
wq.transform = "r"
yv = riskdf.loc[risk_d, "R_now"].values
gm = esda.Moran(yv, wq, permutations=9999)
lm = esda.Moran_Local(yv, wq, permutations=9999, seed=SEED)
qcut = esda.fdr(lm.p_sim, 0.05)
sig = lm.p_sim <= qcut
quad = np.array(["ns"] * len(yv), dtype=object)
labels_q = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
quad[sig] = [labels_q[q] for q in lm.q[sig]]
riskdf.loc[risk_d, "lisa"] = quad
riskdf.loc[risk_d, "lisa_p"] = lm.p_sim
print(f"global Moran's I = {gm.I:.3f} (p = {gm.p_sim:.4f}); islands: {wq.islands}; FDR p-cut = {qcut:.4f}")
print(pd.Series(quad).value_counts().to_string())
print("High-High districts:", riskdf.loc[riskdf.lisa == "High-High", "adm2_name"].tolist())

wealth = R.rwi.reindex(risk_d)
order_w = wealth.rank(method="first").values
frac_rank = (order_w - 0.5) / len(order_w)


def conc_index(loss):
    mu = loss.mean()
    return 2 / mu * np.cov(loss, frac_rank, bias=True)[0, 1] if mu > 0 else np.nan


def poorest20(loss):
    return loss[frac_rank <= 0.2].sum() / loss.sum()


def conc_curve(loss):
    o = np.argsort(frac_rank)
    return np.r_[0, np.arange(1, len(o) + 1) / len(o)], np.r_[0, np.cumsum(loss[o]) / loss.sum()]


Ht = hzi.H_t.values
ineq_rows, curves = [], {}
rb = np.random.default_rng(SEED)
for p in ["present"] + GWLS:
    for metric, vec in [("relative loss (% of trend yield)", H0 if p == "present" else hz.set_index("adm2_pcode").loc[risk_d, f"H_{p}"].values),
                        ("absolute loss (t)", Ht if p == "present" else hz.set_index("adm2_pcode").loc[risk_d, f"H_t_{p}"].values)]:
        ci = conc_index(vec)
        boot_ci = []
        for _ in range(1000):
            ix = rb.integers(0, len(vec), len(vec))
            mu = vec[ix].mean()
            boot_ci.append(2 / mu * np.cov(vec[ix], frac_rank[ix], bias=True)[0, 1])
        gcm_ci = []
        if p != "present":
            arr = Hf_arr[p] if metric.startswith("relative") else Hf_arr[p] * (hzi.trend2025_t_ha * hzi.area_ha).values
            gcm_ci = [conc_index(a) for a in arr]
        ineq_rows.append({"period": "1991-2020" if p == "present" else f"+{p} degC", "metric": metric, "C": ci,
                          "C_boot_lo": np.percentile(boot_ci, 2.5), "C_boot_hi": np.percentile(boot_ci, 97.5),
                          "C_gcm_p10": np.percentile(gcm_ci, 10) if gcm_ci else np.nan,
                          "C_gcm_p90": np.percentile(gcm_ci, 90) if gcm_ci else np.nan,
                          "share_poorest20": poorest20(vec), "share_richest20": vec[frac_rank > 0.8].sum() / vec.sum()})
        curves[(p, metric)] = conc_curve(vec)
ineq = pd.DataFrame(ineq_rows)
ineq.round(4).to_csv(TAB / "T6_inequality.csv", index=False)
print(ineq.round(3).to_string(index=False))
# risk and vulnerability are also checked against wealth
print("Spearman R_now vs RWI:", stats.spearmanr(riskdf.loc[risk_d, "R_now"], wealth))

## 9. Explainable decision support (Stage F)

A gradient-boosted surrogate is fitted to the median present risk using the rescaled hazard, exposure and
the eight oriented vulnerability indicators; TreeSHAP decomposes each district's risk. The indicator with the
largest positive contribution is the dominant driver, mapped to a policy lever.

In [ ]:
SUR = pd.DataFrame({"Heat hazard": Hs(H0), "Exposure": Et}, index=risk_d).join(
    Z.loc[risk_d].rename(columns=VLAB))
sur = xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=2, min_child_weight=3, subsample=0.8,
                       colsample_bytree=0.8, reg_lambda=5, random_state=SEED, n_jobs=N_JOBS)
sur.fit(SUR.values, riskdf.loc[risk_d, "R_now"].values)
sur_r2 = r2_score(riskdf.loc[risk_d, "R_now"], sur.predict(SUR.values))
sur_cv = np.zeros(len(SUR))
for a, b in GroupKFold(n_splits=5).split(SUR, groups=block.reindex(risk_d).fillna(-1).values):
    sur_cv[b] = xgb.XGBRegressor(**sur.get_params()).fit(SUR.values[a], riskdf.loc[risk_d, "R_now"].values[a]).predict(SUR.values[b])
sur_r2_cv = r2_score(riskdf.loc[risk_d, "R_now"], sur_cv)
print(f"surrogate R2 in-sample {sur_r2:.3f}; spatial-block CV R2 {sur_r2_cv:.3f}")
sv_sur = pd.DataFrame(shap.TreeExplainer(sur).shap_values(SUR.values), index=risk_d, columns=SUR.columns)
LEVER = {"Heat hazard": "Heat-tolerant varieties and earlier sowing",
         "Rainfed cultivated area": "Supplementary irrigation and water harvesting",
         "Exposure": "Livelihood and crop diversification",
         "Poverty headcount": "Social protection and crop insurance", "Poverty intensity": "Social protection and crop insurance",
         "Housing deprivation": "Social protection and crop insurance", "Rural illiteracy": "Extension and advisory services",
         "Farms < 5 acres": "Crop insurance and smallholder credit", "Tenant-operated farms": "Crop insurance and smallholder credit",
         "Travel time to city": "Market access and extension"}
riskdf.loc[risk_d, "driver"] = sv_sur.idxmax(axis=1)
riskdf.loc[risk_d, "lever"] = riskdf.loc[risk_d, "driver"].map(LEVER)
top_tercile = riskdf.R_now >= riskdf.R_now.quantile(2 / 3)
typ = (riskdf[top_tercile].groupby("lever").agg(n_districts=("adm2_name", "size"),
                                                 districts=("adm2_name", lambda s: ", ".join(sorted(s))))
       .sort_values("n_districts", ascending=False))
typ.to_csv(TAB / "T7_policy_typology_high_risk.csv")
print(riskdf.driver.value_counts().to_string())
print(typ.to_string())
shap_sur_mean = sv_sur.abs().mean().sort_values(ascending=False)
print("mean |SHAP| of surrogate:\n", shap_sur_mean.round(4).to_string())

## 10. Out-of-sample test: the 2021-22 season and the March-April 2022 heatwave (Stage G)

The 2021-22 season was withheld from detrending, tuning and fitting. (i) The final ensemble predicts the
observed 2022 district anomalies; (ii) districts ranked at high livelihood risk are compared with the yield
losses they recorded (Spearman correlation, Kruskal-Wallis test across risk terciles).

In [ ]:
h22 = data[data.own & (data.year == HOLDOUT_SEASON)].copy()
h22["pred"] = ens_predict(h22[FEATS].values)
h22["pred_xgb"], h22["pred_rf"] = final["XGB"].predict(h22[FEATS].values), final["RF"].predict(h22[FEATS].values)
cl22 = base_sorted.groupby("adm2_pcode")[HEAT_VARS].mean().reindex(h22.adm2_pcode)
h22["heat_loss"] = heat_loss(h22.reset_index(drop=True), cl22.reset_index(drop=True))[0]
w22 = h22.area_ha.fillna(h22.area_ha.median())
g22 = {
    "n_districts": len(h22),
    "observed_mean_anom_pct (area-weighted)": 100 * np.average(h22.anom, weights=w22),
    "predicted_mean_anom_pct (area-weighted)": 100 * np.average(h22.pred, weights=w22),
    "pearson_r": stats.pearsonr(h22.anom, h22.pred)[0], "pearson_p": stats.pearsonr(h22.anom, h22.pred)[1],
    "spearman_rho": stats.spearmanr(h22.anom, h22.pred)[0],
    "RMSE_model": np.sqrt(np.mean((h22.pred - h22.anom) ** 2)), "RMSE_trend": np.sqrt(np.mean(h22.anom ** 2)),
    "share_sign_correct": np.mean(np.sign(h22.pred) == np.sign(h22.anom)),
    "mean_heat_loss_pct": 100 * np.average(h22.heat_loss, weights=w22),
    "heat_loss_1991_2020_mean_pct": 100 * np.average(H0, weights=hzi.area_ha.values),
}
v22 = h22.set_index("adm2_pcode").join(riskdf[["R_now", "H_t", "V_t"]], how="inner")
v22["tercile"] = pd.qcut(v22.R_now, 3, labels=["low", "medium", "high"])
kw = stats.kruskal(*[g.anom.values for _, g in v22.groupby("tercile", observed=False)])
g22.update({"spearman_R_vs_anom": stats.spearmanr(v22.R_now, v22.anom)[0],
            "spearman_R_vs_anom_p": stats.spearmanr(v22.R_now, v22.anom)[1],
            "spearman_H_vs_anom": stats.spearmanr(v22.H_t, v22.anom)[0],
            "spearman_H_vs_anom_p": stats.spearmanr(v22.H_t, v22.anom)[1],
            "kruskal_H": kw.statistic, "kruskal_p": kw.pvalue,
            **{f"mean_anom_pct_{t}_risk": 100 * g.anom.mean() for t, g in v22.groupby("tercile", observed=False)}})
# additional analysis: 2022 losses inside versus outside the significant high-risk cluster (Section 8)
in_hh = v22.index.isin(riskdf.index[riskdf.lisa == "High-High"])
mw = stats.mannwhitneyu(v22.anom[in_hh], v22.anom[~in_hh], alternative="less")
g22.update({"n_highhigh_cluster": int(in_hh.sum()), "mean_anom_pct_highhigh_cluster": 100 * v22.anom[in_hh].mean(),
            "mean_anom_pct_other_districts": 100 * v22.anom[~in_hh].mean(),
            "mannwhitney_U_cluster_lt_other": mw.statistic, "mannwhitney_p_one_sided": mw.pvalue})
val22 = pd.Series(g22)
val22.round(4).to_csv(TAB / "T8_validation_2022.csv", header=["value"])
print(val22.round(3).to_string())
h22.to_csv(TAB / "validation_2022_districts.csv", index=False)

### 10.1 Robustness checks
(a) Hazard ranking with a model that also uses inherited (parent-district) yields; (b) GDHY gridded yields as
an independent cross-check of the compiled district yield series (1982-2016).

In [ ]:
allrows = data[data.year != HOLDOUT_SEASON].reset_index(drop=True)
a_all, _ = refit_anomaly(allrows, np.ones(len(allrows), bool))
ok = np.isfinite(a_all)
m_inh = make_model("XGB", best["XGB"]).fit(allrows.loc[ok, FEATS].values, a_all[ok])
L_inh = heat_loss(base, clim_heat, models=(m_inh, final["RF"]))[0]
H_inh = base.assign(l=L_inh).groupby("adm2_pcode").l.mean().reindex(risk_d)
rob = {"Spearman hazard: own-only vs incl. inherited": stats.spearmanr(H0, H_inh)[0],
       "Spearman hazard: XGB-only vs RF-only": stats.spearmanr(
           base.assign(l=heat_loss(base, clim_heat, models=(final["XGB"],))[0]).groupby("adm2_pcode").l.mean().reindex(risk_d),
           base.assign(l=heat_loss(base, clim_heat, models=(final["RF"],))[0]).groupby("adm2_pcode").l.mean().reindex(risk_d))[0],
       "Spearman risk: PCA vs equal vulnerability weights": stats.spearmanr(
           riskdf.loc[risk_d, "R_now"], Hs(H0) * Et * minmax(R.V_eq).values)[0]}

gdhy = []
for y in range(1982, 2017):
    f = RAW / "04_crop" / "gdhy_wheat" / "wheat" / f"yield_{y}.nc4"
    with xr.open_dataset(f) as ds:
        da = ds["var"].sel(lat=slice(23, 38), lon=slice(60, 79))
        Wg = weight_matrix(da.lat.values, da.lon.values)
        gdhy.append(pd.DataFrame({"adm2_pcode": PCODES, "year": y,
                                  "gdhy": district_mean(Wg, da.values[None].astype("float64"))[0]}))
gdhy = pd.concat(gdhy).merge(tgt[tgt.own][["adm2_pcode", "year", "yield_t_ha"]], on=["adm2_pcode", "year"]).dropna()
gdhy = gdhy[gdhy.gdhy > 0]
rob["Pearson yield vs GDHY (district-seasons, 1982-2016)"] = stats.pearsonr(gdhy.yield_t_ha, gdhy.gdhy)[0]
gdhy["d_obs"] = gdhy.groupby("adm2_pcode").yield_t_ha.diff()
gdhy["d_gdhy"] = gdhy.groupby("adm2_pcode").gdhy.diff()
gg = gdhy.dropna()
rob["Pearson first differences yield vs GDHY"] = stats.pearsonr(gg.d_obs, gg.d_gdhy)[0]
# (c) heat defined more broadly: stage VPD treated as a heat predictor as well
VPD_VARS = [f"vpd_{s_}" for s_ in STAGE_ORDER]
clim_ext = base.groupby("adm2_pcode")[HEAT_VARS + VPD_VARS].transform("mean")
L_ext = heat_loss(base, clim_ext, heat_vars=HEAT_VARS + VPD_VARS)[0]
H_ext = base.assign(l=L_ext).groupby("adm2_pcode").l.mean().reindex(risk_d)
rob["National heat loss %, heat incl. VPD (vs main)"] = 100 * np.average(H_ext, weights=hzi.area_ha.values)
rob["National heat loss %, main definition"] = 100 * np.average(H0, weights=hzi.area_ha.values)
rob["Spearman hazard: main vs heat incl. VPD"] = stats.spearmanr(H0, H_ext)[0]
rob_tab = pd.Series(rob)
rob_tab.round(4).to_csv(TAB / "S2_robustness.csv", header=["value"])
print(rob_tab.round(3).to_string())

In [ ]:
# final district table (risk, uncertainty, LISA cluster, dominant driver and lever) and surrogate SHAP values
riskdf.round(5).to_csv(TAB / "district_risk.csv")
sv_sur.round(5).to_csv(TAB / "district_risk_drivers_shap.csv")
print("district tables written:", len(riskdf), "districts")

## 11. Figures for the manuscript
Figures 3-9 (Figures 1-2, study area and framework, were produced earlier by `scripts/01_*` and `02_*`).
All maps use the 2022 OCHA boundaries; districts outside the risk analysis are shown in light grey.

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

prov = adm.dissolve("adm1_name").reset_index()
gplot = adm.set_index("adm2_pcode").copy()
gplot["geometry"] = gplot.geometry.simplify(0.01, preserve_topology=True)  # ~1 km, keeps vector PDFs small
prov["geometry"] = prov.geometry.simplify(0.01, preserve_topology=True)
W2 = 7.16  # IEEE double-column width (in)


def base_map(ax):
    gplot.plot(ax=ax, color="#eeeeee", edgecolor="#c8c8c8", linewidth=0.15)
    ax.set_xlim(60.6, 77.9)
    ax.set_ylim(23.6, 37.2)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_linewidth(0.4)


def map_values(ax, series, cmap, vmin, vmax, title, cblabel, fmt="%.2f", extend="neither"):
    base_map(ax)
    g = gplot.loc[series.index].assign(v=series.values)
    g.plot(column="v", cmap=cmap, vmin=vmin, vmax=vmax, ax=ax, edgecolor="#8a8a8a", linewidth=0.15)
    prov.boundary.plot(ax=ax, color="#1a1a1a", linewidth=0.45)
    sm_ = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
    cb = plt.colorbar(sm_, ax=ax, orientation="horizontal", fraction=0.045, pad=0.02, aspect=26, extend=extend,
                      format=fmt)
    cb.ax.tick_params(labelsize=6, width=0.4, length=2)
    cb.outline.set_linewidth(0.4)
    cb.set_label(cblabel, fontsize=6.5)
    ax.set_title(title, loc="left", fontsize=7.5)


def top_legend(ax, frac=0.24, ncol=3, **kw):
    """Single-row legend in headroom added above the data, so it never covers bars, lines or points."""
    y0, y1 = ax.get_ylim()
    ax.set_ylim(y0, y1 + frac * (y1 - y0))
    ax.legend(frameon=False, loc="upper center", ncol=ncol, fontsize=kw.pop("fontsize", 6), handlelength=1.4,
              columnspacing=1.0, borderaxespad=0.2, **kw)


def below_legend(ax, ncol=3, offset=-0.04, **kw):
    """Legend placed outside the axes, under the panel."""
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, offset), ncol=ncol,
              fontsize=kw.pop("fontsize", 6), handlelength=1.2, columnspacing=0.9, **kw)


def save(fig, name):
    # editable text in vector files: TrueType fonts in PDF, real text (not paths) in SVG
    plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"})
    fig.savefig(FIG / f"{name}.pdf")
    fig.savefig(FIG / f"{name}.png", dpi=300)
    edit = FIG / "editable"
    edit.mkdir(exist_ok=True)
    fig.savefig(edit / f"{name}.pdf")
    fig.savefig(edit / f"{name}.svg")
    fig.savefig(edit / f"{name}.png", dpi=600)
    fig.savefig(edit / f"{name}.tif", dpi=600, pil_kwargs={"compression": "tiff_lzw"})
    plt.show()
    plt.close(fig)


# ---------------- Fig. 3: data and model skill
fig = plt.figure(figsize=(W2, 2.45), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 1, 1])
ax = fig.add_subplot(gs[0])
nat = panel[panel.own].groupby("year").apply(lambda g: g.prod_t.sum() / g.area_ha.sum())
ann = tgt[tgt.own].groupby("year").apply(lambda g: np.average(g.anom, weights=g.area_ha.fillna(g.area_ha.median())))
ax.bar(ann.index, 100 * ann.values, color=np.where(ann.values < 0, "#c0392b", "#2e86c1"), width=0.8)
ax.set_ylabel("Area-weighted yield anomaly (%)")
ax2 = ax.twinx()
ax2.plot(nat.index, nat.values, color="k", lw=0.9)
ax2.set_ylabel("Yield of reporting districts (t/ha)")
ax.axvline(HOLDOUT_SEASON, color="#555", lw=0.6, ls="--")
ax.set_title("(a) Detrended district yield anomalies", loc="left")
ax = fig.add_subplot(gs[1])
sk = skill[skill.model != "Trend"].pivot(index="design", columns="model", values="SS_vs_trend").loc[
    ["LOYO", "Spatial blocks", "Forward chaining"], ["Ridge", "RF", "XGB"]]
xs = np.arange(len(sk))
for i, (m, c) in enumerate(zip(sk.columns, ["#95a5a6", "#27ae60", "#8e44ad"])):
    ax.bar(xs + (i - 1) * 0.26, sk[m].values, width=0.25, color=c, label=m)
ax.axhline(0, color="k", lw=0.5)
ax.set_xticks(xs)
ax.set_xticklabels(["LOYO", "Spatial\nblocks", "Forward\nchaining"])
ax.set_ylabel("Skill score vs trend baseline")
top_legend(ax, frac=0.28)
ax.set_title("(b) Out-of-sample skill", loc="left")
ax = fig.add_subplot(gs[2])
lo = oof[oof.design == "LOYO"]
ax.hexbin(100 * lo.obs, 100 * lo.pred_XGB, gridsize=35, cmap="Greys", mincnt=1, linewidths=0.1)
ax.plot([-70, 70], [-70, 70], color="#c0392b", lw=0.6)
r_ = stats.pearsonr(lo.obs, lo.pred_XGB)[0]
ax.text(0.04, 0.93, f"r = {r_:.2f}, n = {len(lo)}", transform=ax.transAxes, fontsize=6.5)
ax.set_xlim(-70, 70)
ax.set_ylim(-45, 45)
ax.set_xlabel("Observed anomaly (%)")
ax.set_ylabel("Predicted anomaly, XGB (%)")
ax.set_title("(c) LOYO predictions", loc="left")
save(fig, "Fig3_yield_and_skill")

# ---------------- Fig. 4: SHAP and thresholds
fig = plt.figure(figsize=(W2, 4.3), constrained_layout=True)
gs = fig.add_gridspec(2, 3)
ax = fig.add_subplot(gs[0, 0])
hm = imp[imp.stage != "-"].pivot_table(index="variable", columns="stage", values="mean_abs_shap_XGB").reindex(
    columns=STAGE_ORDER)
hm = hm.loc[hm.sum(1).sort_values(ascending=False).index] * 100
im = ax.imshow(hm.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(5))
ax.set_xticklabels(["Sow", "Till", "Joint", "Anth", "Fill"])
ax.set_yticks(range(len(hm)))
ax.set_yticklabels([{"tmax": "Tmax", "kdd30": "KDD30", "kdd34": "KDD34", "gdd": "GDD", "frost": "Frost days",
                     "vpd": "VPD", "rain": "Rainfall", "dry": "Dry spell", "sm": "Soil moisture"}.get(v, v)
                    for v in hm.index])
cb = plt.colorbar(im, ax=ax, fraction=0.05, pad=0.02)
cb.set_label("Mean |SHAP| (pp)", fontsize=6.5)
cb.ax.tick_params(labelsize=6)
ax.set_title("(a) Predictor importance (XGB)", loc="left")
for k, s in enumerate(["joint", "anth", "fill"]):
    ax = fig.add_subplot(gs[0, 1 + k] if k < 2 else gs[1, 0])
    j = FEATS.index(f"tmax_{s}")
    ax.scatter(Xd[:, j], 100 * sv_x[:, j], s=1.2, c=dv.irr_share.values, cmap="viridis_r", alpha=0.5, lw=0,
               rasterized=True)
    t_ = thr_x.set_index("stage").loc[s]
    _, lo_ = zero_cross(Xd[:, j], sv_x[:, j])
    ax.plot(lo_[:, 0], 100 * lo_[:, 1], color="#c0392b", lw=1)
    ax.axhline(0, color="k", lw=0.4)
    if np.isfinite(t_.threshold_C) and t_.pct_obs_above >= 1:  # thresholds exceeded by <1 % of seasons are not shown
        ax.axvspan(t_.ci_lo, t_.ci_hi, color="#c0392b", alpha=0.12, lw=0)
        ax.axvline(t_.threshold_C, color="#c0392b", lw=0.6, ls="--")
        ax.text(0.03, 0.06, f"threshold {t_.threshold_C:.1f} °C\n[{t_.ci_lo:.1f}, {t_.ci_hi:.1f}]",
                transform=ax.transAxes, fontsize=6.3)
    ax.set_xlabel(f"Mean Tmax, {dict(joint='jointing (Feb)', anth='anthesis (Mar)', fill='grain filling (Apr)')[s]} (°C)")
    ax.set_ylabel("SHAP (pp of yield)")
    ax.set_title(f"({'bcd'[k]}) SHAP dependence", loc="left")
ax = fig.add_subplot(gs[1, 1])
for s, c in zip(["joint", "anth", "fill"], ["#2e86c1", "#e67e22", "#c0392b"]):
    e, a = ale_res[s]
    ax.plot(e, 100 * a, color=c, lw=1, label={"joint": "Feb", "anth": "Mar", "fill": "Apr"}[s])
ax.axhline(0, color="k", lw=0.4)
ax.set_xlabel("Stage mean Tmax (°C)")
ax.set_ylabel("ALE (pp of yield)")
top_legend(ax, frac=0.22)
ax.set_title("(e) Accumulated local effects", loc="left")
ax = fig.add_subplot(gs[1, 2])
for v, c in [("kdd34_fill", "#c0392b"), ("kdd34_anth", "#e67e22"), ("kdd30_fill", "#7f8c8d")]:
    j = FEATS.index(v)
    lo_ = lowess(sv_x[:, j], Xd[:, j], frac=0.4)
    ax.plot(lo_[:, 0], 100 * lo_[:, 1], color=c, lw=1, label=v.upper().replace("_FILL", " Apr").replace("_ANTH", " Mar"))
ax.axhline(0, color="k", lw=0.4)
ax.set_xlabel("Killing degree days (°C·d)")
ax.set_ylabel("SHAP (pp of yield)")
top_legend(ax, frac=0.36, ncol=2, fontsize=5.6)
ax.set_title("(f) Response to extreme heat", loc="left")
save(fig, "Fig4_shap_thresholds")

# ---------------- Fig. 5: hazard now and at warming levels
fig, axs = plt.subplots(1, 4, figsize=(W2, 2.35), constrained_layout=True)
hzz = hz.set_index("adm2_pcode")
vmax_h = np.nanpercentile(100 * hzz[[f"H_{g}" for g in GWLS]].values, 97)
for ax, col, ttl in zip(axs, ["H"] + [f"H_{g}" for g in GWLS],
                        ["(a) 1991–2020", "(b) +1.5 °C", "(c) +2 °C", "(d) +3 °C"]):
    map_values(ax, 100 * hzz[col], "OrRd", 0, vmax_h, ttl, "Heat loss (% of trend yield)", fmt="%.1f", extend="max")
save(fig, "Fig5_hazard_warming")

# ---------------- Fig. 6: exposure, vulnerability, risk, uncertainty
fig, axs = plt.subplots(1, 4, figsize=(W2, 2.35), constrained_layout=True)
map_values(axs[0], riskdf.E_t, "BuPu", 0, 1, "(a) Exposure", "E (0–1)")
map_values(axs[1], riskdf.V_t, "PuRd", 0, 1, "(b) Vulnerability", "V (0–1)")
map_values(axs[2], riskdf.R_now, "inferno_r", 0, np.nanpercentile(riskdf.R_now, 98), "(c) Livelihood risk",
           "Median R", extend="max")
map_values(axs[3], riskdf.rank_now_hi - riskdf.rank_now_lo, "Greys", 0, np.nanpercentile(riskdf.rank_now_hi - riskdf.rank_now_lo, 98),
           "(d) Rank uncertainty", "90% rank interval width", fmt="%.0f")
save(fig, "Fig6_exposure_vulnerability_risk")

# ---------------- Fig. 7: hotspots and inequality
fig = plt.figure(figsize=(W2, 2.5), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.05, 1, 1])
ax = fig.add_subplot(gs[0])
base_map(ax)
cols = {"High-High": "#b2182b", "Low-Low": "#2166ac", "High-Low": "#f4a582", "Low-High": "#92c5de", "ns": "#f7f7f7"}
gl = gplot.loc[risk_d].assign(c=riskdf.loc[risk_d, "lisa"].map(cols).values)
gl.plot(color=gl.c, ax=ax, edgecolor="#8a8a8a", linewidth=0.15)
prov.boundary.plot(ax=ax, color="#1a1a1a", linewidth=0.45)
ax.legend(handles=[Patch(fc=v, ec="#8a8a8a", lw=0.3, label=k if k != "ns" else "not significant")
                   for k, v in cols.items()], frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.01),
          ncol=3, fontsize=5.8, handlelength=1.2, columnspacing=0.9)
ax.set_title(f"(a) Risk hotspots (LISA, FDR 5%), I = {gm.I:.2f}", loc="left")
ax = fig.add_subplot(gs[1])
for p, c in zip(["present"] + GWLS, ["k", "#f39c12", "#e74c3c", "#7b241c"]):
    x_, y_ = curves[(p, "relative loss (% of trend yield)")]
    ax.plot(x_, y_, color=c, lw=0.9, label="1991–2020" if p == "present" else f"+{p} °C")
ax.plot([0, 1], [0, 1], color="#999", lw=0.5, ls="--")
ax.set_xlabel("Cumulative share of districts,\npoorest to richest (RWI)")
ax.set_ylabel("Cumulative share of heat loss")
ax.legend(frameon=False, loc="upper left", fontsize=6)
ax.set_title("(b) Concentration curves", loc="left")
ax = fig.add_subplot(gs[2])
iq = ineq[ineq.metric.str.startswith("relative")]
xs = np.arange(len(iq))
ax.bar(xs - 0.18, 100 * iq.share_poorest20, width=0.35, color="#7b241c", label="Poorest 20%")
ax.bar(xs + 0.18, 100 * iq.share_richest20, width=0.35, color="#5d6d7e", label="Richest 20%")
ax.axhline(20, color="#999", lw=0.5, ls="--")
ax.set_xticks(xs)
ax.set_xticklabels(["1991–\n2020", "+1.5 °C", "+2 °C", "+3 °C"])
ax.set_ylabel("Share of total heat loss (%)")
ytop = 100 * max(iq.share_poorest20.max(), iq.share_richest20.max())
for x_, c_ in zip(xs, iq.C):
    top_bar = 100 * max(iq.share_poorest20.iloc[x_], iq.share_richest20.iloc[x_])
    ax.text(x_, top_bar + 0.8, f"C = {c_:.2f}", ha="center", va="bottom", fontsize=6)
ax.set_ylim(0, ytop + 11)
ax.legend(frameon=False, ncol=2, loc="upper center", fontsize=6, handlelength=1.4, borderaxespad=0.2)
ax.set_title("(c) Loss shares by wealth quintile", loc="left")
save(fig, "Fig7_hotspots_inequality")

# ---------------- Fig. 8: policy typology
fig = plt.figure(figsize=(W2, 2.9), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[1.35, 1])
ax = fig.add_subplot(gs[0])
base_map(ax)
levers = sorted(riskdf.lever.dropna().unique())
pal = dict(zip(levers, ["#c0392b", "#2980b9", "#27ae60", "#8e44ad", "#f39c12", "#16a085", "#7f8c8d", "#d35400"]))
gt = gplot.loc[risk_d].assign(lever=riskdf.loc[risk_d, "lever"].values, high=top_tercile.reindex(risk_d).values)
for lv, gsub in gt.groupby("lever"):
    if gsub.high.any():
        gsub[gsub.high].plot(ax=ax, color=pal[lv], edgecolor="#555", linewidth=0.2)
    if (~gsub.high).any():
        gsub[~gsub.high].plot(ax=ax, color=pal[lv], alpha=0.35, edgecolor="#999", linewidth=0.15)
prov.boundary.plot(ax=ax, color="#1a1a1a", linewidth=0.45)
ax.legend(handles=[Patch(fc=pal[l], label=l) for l in levers], frameon=False, loc="upper center",
          bbox_to_anchor=(0.5, -0.01), ncol=2, fontsize=5.8)
ax.set_title("(a) Dominant risk driver and policy lever (solid: top risk tercile)", loc="left")
ax = fig.add_subplot(gs[1])
sm_ = shap_sur_mean.iloc[::-1]
ax.barh(range(len(sm_)), sm_.values, color="#34495e")
ax.set_yticks(range(len(sm_)))
ax.set_yticklabels(sm_.index)
ax.set_xlabel("Mean |SHAP| on risk index")
ax.set_title(f"(b) Drivers of risk (surrogate CV R² = {sur_r2_cv:.2f})", loc="left")
save(fig, "Fig8_policy_typology")

# ---------------- Fig. 9: 2022 validation
fig = plt.figure(figsize=(W2, 2.45), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.9, 1.05])
ax = fig.add_subplot(gs[0])
cprov = {"Punjab": "#2e86c1", "Sindh": "#e67e22", "Khyber Pakhtunkhwa": "#27ae60", "Balochistan": "#8e44ad", "Islamabad": "#7f8c8d"}
for pv, g in h22.groupby("adm1_name"):
    ax.scatter(100 * g.anom, 100 * g.pred, s=8, color=cprov[pv], label=pv, lw=0)
ax.plot([-60, 40], [-60, 40], color="#999", lw=0.5, ls="--")
ax.axhline(0, color="k", lw=0.3)
ax.axvline(0, color="k", lw=0.3)
ax.set_xlabel("Observed 2022 anomaly (%)")
ax.set_ylabel("Predicted 2022 anomaly (%)")
ax.text(0.03, 0.93, f"r = {g22['pearson_r']:.2f}", transform=ax.transAxes, fontsize=6.5)
# the lower-right quadrant (observed > 0, predicted < -25 %) holds no districts: legend goes there
assert (100 * h22.pred).min() > -25
ax.legend(frameon=False, fontsize=5.5, loc="lower right", borderaxespad=0.3, handletextpad=0.2)
ax.set_title("(a) Hazard model, 2021–22 hold-out", loc="left")
ax = fig.add_subplot(gs[1])
data_t = [100 * g.anom.values for _, g in v22.groupby("tercile", observed=False)]
bp = ax.boxplot(data_t, widths=0.55, patch_artist=True, medianprops=dict(color="k", lw=0.8),
                flierprops=dict(markersize=2))
for patch, c in zip(bp["boxes"], ["#fdebd0", "#f5b041", "#ba4a00"]):
    patch.set_facecolor(c)
    patch.set_linewidth(0.5)
ax.set_xticklabels(["Low", "Medium", "High"])
ax.set_xlabel("Livelihood-risk tercile")
ax.set_ylabel("Observed 2022 anomaly (%)")
ax.axhline(0, color="k", lw=0.3)
y0, y1 = ax.get_ylim()
ax.set_ylim(y0 - 0.25 * (y1 - y0), y1)   # room below the boxes for the cluster note
ax.set_title(f"(b) Risk terciles (Kruskal–Wallis p = {g22['kruskal_p']:.2f})", loc="left")
ax.text(0.03, 0.04, f"LISA high-risk cluster: {g22['mean_anom_pct_highhigh_cluster']:.1f}% vs "
        f"{g22['mean_anom_pct_other_districts']:.1f}%" + chr(10) + f"(Mann–Whitney one-sided p = {g22['mannwhitney_p_one_sided']:.3f})",
        transform=ax.transAxes, fontsize=5.8)
ax = fig.add_subplot(gs[2])
map_values(ax, 100 * h22.set_index("adm2_pcode").anom, "RdBu", -40, 40, "(c) Observed 2022 anomaly", "% of trend yield",
           fmt="%.0f", extend="both")
gplot.loc[riskdf.index[riskdf.lisa == "High-High"]].boundary.plot(ax=ax, color="#00a000", linewidth=0.9)
ax.text(61.0, 36.4, "green outline:" + chr(10) + "high-risk cluster", fontsize=5.8, color="#007000", va="top")
save(fig, "Fig9_validation_2022")
print("figures written to", FIG)

## 12. Manuscript tables (LaTeX)
Result tables are written as booktabs LaTeX fragments to `results/tables/latex/`, so that every number in the
manuscript comes from this notebook.

In [ ]:
TEX = TAB / "latex"
TEX.mkdir(exist_ok=True)


def tex_table(df, path, fmt):
    lines = []
    for _, r in df.iterrows():
        cells = [fmt.get(c, "{}").format(r[c]) if pd.notna(r[c]) else "--" for c in df.columns]
        lines.append(" & ".join(cells) + r" \\")
    (TEX / path).write_text("\n".join(lines).replace("%", r"\%").replace("_", r"\_"), encoding="utf-8")


t2 = skill.copy()
t2["design"] = pd.Categorical(t2.design, ["LOYO", "Spatial blocks", "Forward chaining"])
t2["model"] = pd.Categorical(t2.model, MODELS)
t2 = t2.sort_values(["design", "model"])
tex_table(t2[["design", "model", "RMSE_anom", "R2_anom", "RMSE_yield_t_ha", "rRMSE_yield_pct", "SS_vs_trend"]], "T2_skill.tex",
          {"RMSE_anom": "{:.3f}", "R2_anom": "{:.3f}", "RMSE_yield_t_ha": "{:.3f}", "rRMSE_yield_pct": "{:.1f}",
           "SS_vs_trend": "{:.3f}"})
tex_table(thr[thr.model == "XGB"][["stage", "threshold_C", "ci_lo", "ci_hi", "slope_pp_per_C_above", "pct_obs_above",
                                   "ALE_threshold_C"]], "T3_thresholds.tex",
          {c: "{:.1f}" for c in ["threshold_C", "ci_lo", "ci_hi", "slope_pp_per_C_above", "pct_obs_above", "ALE_threshold_C"]})
tex_table(fut_tab, "T4_hazard.tex", {"n_models": "{:.0f}", "loss_pct_trend_yield": "{:.2f}", "gcm_p10": "{:.2f}",
                                     "gcm_p90": "{:.2f}", "loss_kt": "{:.0f}"})
tex_table(pca_tab[["indicator", "weight_PCA", "loading_PC1", "loading_PC2"]], "T5_weights.tex",
          {"weight_PCA": "{:.3f}", "loading_PC1": "{:.2f}", "loading_PC2": "{:.2f}"})
tex_table(ineq[["period", "metric", "C", "C_boot_lo", "C_boot_hi", "share_poorest20", "share_richest20"]], "T6_inequality.tex",
          {"C": "{:.3f}", "C_boot_lo": "{:.3f}", "C_boot_hi": "{:.3f}", "share_poorest20": "{:.3f}", "share_richest20": "{:.3f}"})
print(sorted(os.listdir(TEX)))